## 1. 安裝 HyperFlow H3 Flash 環境


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch

'''
Keep services in Kaggle's writable workspace.
將服務程式放在 Kaggle 可寫入的工作空間。
'''
ROOT = Path('/kaggle/working')

'''
Isolate HyperFlow from the previous MiniMax H3 checkout.
將 HyperFlow 與舊版 MiniMax H3 程式環境隔離。
'''
COMFY = ROOT / 'ComfyUI-HyperFlowH3'

'''
Keep large model files outside Kaggle's output directory.
將大型模型權重存放在 Kaggle 暫存區。
'''
STORE = Path('/tmp/hyperflow_h3_models')

'''
Avoid conflicting with the legacy MiniMax H3 local service.
避免與舊版 MiniMax H3 的本機服務連接埠衝突。
'''
PORT = 8194

'''
Use the verified ComfyUI release with native MiniMax-H3 support.
使用已確認具備原生 MiniMax H3 支援的 ComfyUI 正式版。
'''
COMFY_REF = 'v0.45.0'

'''
Require two GPUs without claiming their memory is unified.
確認有兩張 GPU，但不把兩張卡的 VRAM 視為可直接合併。
'''
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, '請啟用 Kaggle T4 x2。'

'''
Display actual hardware before model selection.
在選擇模型前確認實際硬體容量。
'''
for i in range(2):

    '''
    Report each CUDA device and usable VRAM.
    顯示各張 CUDA 裝置及 VRAM 容量。
    '''
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} / {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB')

'''
Install the official HyperFlow loader and the 8B/4B projection runtime.
安裝官方 HyperFlow Loader 與 8B／4B 投影節點。
'''
repositories = {
    COMFY: 'https://github.com/Comfy-Org/ComfyUI.git',
    COMFY / 'custom_nodes/ComfyUI-GGUF': 'https://github.com/city96/ComfyUI-GGUF.git',
    COMFY / 'custom_nodes/ComfyUI-HyperFlow-Flash': 'https://github.com/Video-Rebirth/ComfyUI-HyperFlow-Flash.git',
    COMFY / 'custom_nodes/ComfyUI-ClipProj': 'https://github.com/nicolab28/ComfyUI-ClipProj.git',
}

'''
Detect source revisions changed since a previous notebook execution.
偵測重新執行 Notebook 期間是否有更新來源版本。
'''
H3_RESTART_REQUIRED = False

'''
Clone missing repositories and update only unmodified node sources.
複製缺少的倉庫，僅自動更新沒有本地修改的節點程式。
'''
for directory, url in repositories.items():

    '''
    Reject unknown directories instead of overwriting user data.
    拒絕來源不明的資料夾，避免覆寫使用者檔案。
    '''
    assert not directory.exists() or (directory / '.git').is_dir(), f'不是 Git 倉庫：{directory}'

    '''
    Reuse a clean Git checkout without downloading its weights again.
    沿用乾淨的 Git 倉庫，不重複下載模型權重。
    '''
    if (directory / '.git').is_dir():

        '''
        Refuse to overwrite manually modified plugin code.
        避免覆寫手動修改過的外掛原始碼。
        '''
        dirty = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True).strip()

        '''
        Fetch updates only when the custom node has no modifications.
        只有節點程式沒有本地修改時才取得更新。
        '''
        if not dirty and directory != COMFY:

            '''
            Keep plugin changes within fast-forward history.
            僅允許外掛進行 fast-forward 更新。
            '''
            before = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()

            '''
            Pull the upstream plugin changes.
            拉取上游外掛更新。
            '''
            subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', '-q'], check=True)

            '''
            Restart the server if any node source file changed.
            若節點原始碼已變更，就要求重新啟動服務。
            '''
            H3_RESTART_REQUIRED |= before != subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()

        elif dirty:

            '''
            Notify the user when an existing checkout was left unchanged.
            若倉庫有本地修改而保留不更新，明確告知使用者。
            '''

            '''
            Preserve local changes for manual inspection.
            保留本地修改供使用者自行檢查。
            '''
            print(f'本地修改未覆寫：{directory.name}')

    else:

        '''
        Install repositories that do not exist yet.
        安裝尚不存在的程式庫。
        '''

        '''
        Ensure the Git checkout's parent folder exists.
        確保 Git 目標資料夾的上層路徑存在。
        '''
        directory.parent.mkdir(parents=True, exist_ok=True)

        '''
        Pin only the core ComfyUI checkout to a release tag.
        只有 ComfyUI 核心固定於正式版標籤。
        '''
        options = ['--branch', COMFY_REF] if directory == COMFY else []

        '''
        Fetch code rather than copying previous H3 installation files.
        下載原始碼，不混用舊版 H3 的安裝目錄。
        '''
        subprocess.run(['git', 'clone', '--depth', '1', *options, url, str(directory)], check=True)

        '''
        Mark the service for refresh after adding a node.
        安裝新節點後要求服務重新載入。
        '''
        H3_RESTART_REQUIRED = True

'''
Fetch the selected release on every run.
每次執行都確認指定的 ComfyUI 正式版。
'''
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', 'tag', COMFY_REF], check=True)

'''
Record current core version before selecting the release.
切換正式版前先記錄目前核心版本。
'''
previous_core = subprocess.check_output(['git', '-C', str(COMFY), 'rev-parse', 'HEAD'], text=True).strip()

'''
Apply the ComfyUI release instead of an untested HEAD.
使用 ComfyUI 正式發行版本，而非未驗證的最新開發分支。
'''
subprocess.run(['git', '-C', str(COMFY), 'checkout', '-q', '--detach', COMFY_REF], check=True)

'''
Restart if an existing process is using an older core checkout.
如果執行中的 ComfyUI 核心版本過舊，標記為需要重新啟動。
'''
H3_RESTART_REQUIRED |= previous_core != subprocess.check_output(['git', '-C', str(COMFY), 'rev-parse', 'HEAD'], text=True).strip()

'''
Keep Kaggle's installed PyTorch and CUDA combination unchanged.
保留 Kaggle 原本安裝的 PyTorch 與 CUDA 組合。
'''
constraint = ROOT / 'hyperflow_torch_constraints.txt'

'''
Avoid replacing the running kernel's PyTorch binaries.
避免 pip 更換目前 Kernel 的 PyTorch 二進位套件。
'''
constraint.write_text(f'torch=={torch.__version__}\n', encoding='utf-8')

'''
Install only the dependencies declared by ComfyUI and HyperFlow.
僅安裝 ComfyUI 與 HyperFlow 官方宣告的相依套件。
'''
requirements = [
    COMFY / 'requirements.txt',
    COMFY / 'custom_nodes/ComfyUI-GGUF/requirements.txt',
    COMFY / 'custom_nodes/ComfyUI-HyperFlow-Flash/requirements.txt',
]

'''
Use the same Python environment as the notebook kernel.
確保套件安裝在 Notebook Kernel 使用的 Python 環境。
'''
for requirement in requirements:

    '''
    Skip an optional file that is not present in this release.
    如果某版沒有相依套件檔案，就安全略過。
    '''
    if requirement.is_file():

        '''
        Install with the installed PyTorch version constrained.
        安裝依賴時固定目前 PyTorch 版本。
        '''
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(requirement)], check=True)

'''
Install runtime utilities for local model downloads and progress monitoring.
安裝本機下載權重、通訊與進度監控工具。
'''
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint),
                'huggingface_hub', 'hf_xet', 'requests', 'websocket-client', 'tqdm', 'ipywidgets>=8,<9'], check=True)

'''
Display installed source revisions for reproducibility.
顯示各程式庫的實際版本，方便問題追蹤。
'''
for directory in repositories:

    '''
    Print each checkout's abbreviated commit hash.
    列出每個程式庫的簡短 commit 版本。
    '''
    revision = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', '--short', 'HEAD'], text=True).strip()

    '''
    Report the installed checkout identity.
    顯示目前安裝的程式碼版本。
    '''
    print(f'{directory.name}: {revision}')

'''
Confirm the installation phase is complete.
確認環境安裝階段已完成。
'''
print('✓ 第 1 節完成：HyperFlow H3 Flash / ComfyUI v0.45.0')

## 2. 下載 HyperFlow 模型與輔助權重


In [ ]:
'''
Check disk capacity before downloading gigabyte-scale files.
下載大型權重前先檢查可用磁碟空間。
'''
import shutil

'''
Keep Hugging Face metadata and cache in temporary storage before importing it.
在匯入 Hugging Face 套件前設定快取位置，確保環境變數生效。
'''
os.environ['HF_HOME'] = '/tmp/hyperflow_h3_hf'

'''
Avoid writing duplicate model-cache files to /kaggle/working.
避免在 Kaggle 工作資料夾複製大型模型快取。
'''
os.environ['HF_HUB_CACHE'] = '/tmp/hyperflow_h3_hf/hub'

'''
Disable the optional extra Xet chunk cache.
停用不必要的 Xet 分塊快取以節省空間。
'''
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'

'''
Download only the requested model files.
僅下載目前選擇的模型檔案。
'''
from huggingface_hub import hf_hub_download

'''
Choose the generation task before model download; select the same task for generation.
在下載模型前先選擇任務；生成 Cell 會沿用此設定。 Options / 可選值: 't2v' 文字生成、'i2v' 起始圖片、'flf2v' 首尾圖片、'r2v' 多張參考圖片。
'''
TASK = 't2v'

'''
Validate the selected task before requesting large model files.
下載大型權重前先確認任務名稱有效。
'''
assert TASK in ('t2v', 'i2v', 'flf2v', 'r2v'), f'不支援的 TASK：{TASK}'

'''
Select the compressed FL2VA or the separate Ref2VA diffusion architecture.
一般影片使用 FL2VA，多圖參考 R2V 則使用獨立的 Ref2VA。
'''
MODEL_FAMILY = 'ref2va' if TASK == 'r2v' else 'fl2va'

'''
Look up task-specific model settings if its parameter cell has already run.
如果對應任務的參數 Cell 已執行，優先採用該任務的模型設定。
'''
task_models = globals().get('TASK_CONFIGS', {}).get(TASK, {})

'''
Select GPU 0 quantization, with a conservative Ref2VA first-run default.
設定 GPU 0 精度；R2V 首次執行預設較省記憶體的 Q4_K_M。
'''
QUANT = task_models.get('quant', 'Q4_K_M' if TASK == 'r2v' else 'Q8_0')

'''
Prefer the native visual encoder for reference images; ordinary modes retain 8B FP8.
R2V 預設使用原生 32B 視覺編碼器；其餘模式維持 8B FP8。
'''
ENCODER_VARIANT = task_models.get('encoder_variant', '32B_NVFP4' if TASK == 'r2v' else '8B_FP8')

'''
Identify whether the encoder uses the native 32B tower or ClipProj.
判斷 Encoder 使用原生 32B，或透過 ClipProj 投影的 8B／4B。
'''
ENCODER_SIZE = ENCODER_VARIANT.split('_', 1)[0]

'''
Map published GGUF sizes for the two independent diffusion architectures (decimal GB).
記錄 FL2VA 與 Ref2VA 兩套獨立模型的公開權重容量（十進位 GB）。
'''
MODEL_SIZES_GB = {
    'fl2va': {'Q3_K_M': 7.22, 'Q4_K_S': 8.01, 'Q4_K_M': 9.05,
              'Q5_K_M': 10.3, 'Q6_K': 11.6, 'Q8_0': 14.9},
    'ref2va': {'Q3_K_M': 7.4, 'Q4_K_S': 8.14, 'Q4_K_M': 9.22,
               'Q5_K_M': 10.45, 'Q6_K': 11.76, 'Q8_0': 15.16},
}

'''
Use the size table corresponding to the selected task's transformer.
依目前任務的 Transformer 選取正確的權重大小表。
'''
QUANT_SIZES_GB = MODEL_SIZES_GB[MODEL_FAMILY]

'''
Validate GPU 0's published GGUF quantization.
確認 GPU 0 選用官方實際提供的 GGUF 量化。
'''
assert QUANT in QUANT_SIZES_GB, f'GPU 0 不支援 {QUANT}。'

'''
Obtain the specific compressed HyperFlow checkpoint, not the old 31B GGUF.
下載真正的 HyperFlow 壓縮模型，而不是先前 31B 的 GGUF。
'''
GGUF_REPO = 'VideoRebirthAI/hyperflow-h3-flash'

'''
Derive the published model filename from the quantization.
使用官方命名規則建立所選量化檔名。
'''
DIT_NAME = f'minimax_h3_{MODEL_FAMILY}_hyperflow_flash_14b-{QUANT}.gguf'

'''
Map GPU 1 choices to their real Hugging Face weights, not invented GGUF labels.
將 GPU 1 選項對應到 Hugging Face 真實權重，不使用不存在的 GGUF 名稱。
'''
ENCODERS = {
    '32B_NVFP4': ('Comfy-Org/MiniMax-H3', 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 15.7),
    '32B_INT8': ('Comfy-Org/MiniMax-H3', 'qwen3vl_32b_minimax_h3_int8_convrot.safetensors', 27.1),
    '32B_BF16': ('Comfy-Org/MiniMax-H3', 'qwen3vl_32b_minimax_h3_bf16.safetensors', 51.5),
    '8B_NVFP4': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_nvfp4.safetensors', 6.31),
    '8B_W4A8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_w4a8.safetensors', 6.31),
    '8B_INT8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_int8_convrot.safetensors', 9.35),
    '8B_FP8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_fp8_scaled.safetensors', 10.6),
    '8B_BF16': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_bf16.safetensors', 17.5),
    '4B_FP8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_4b_fp8_scaled.safetensors', 5.24),
    '4B_BF16': ('Comfy-Org/Qwen3-VL', 'qwen3vl_4b_bf16.safetensors', 8.88),
}

'''
Stop before downloading an unsupported GPU 1 model variant.
若 GPU 1 模型選項不存在，下載前就明確停止。
'''
assert ENCODER_VARIANT in ENCODERS, f'GPU 1 不支援 {ENCODER_VARIANT}；可選：{list(ENCODERS)}'

'''
Find the exact repository and checkpoint for the chosen encoder.
取得所選 Encoder 的正確來源、檔名與權重大小。
'''
ENCODER_REPO, encoder_name, ENCODER_GB = ENCODERS[ENCODER_VARIANT]

'''
Use the documented model directory rather than guessing the file path.
使用官方存放位置，而非自行推測檔案路徑。
'''
ENCODER_FILE = f'text_encoders/{encoder_name}'

'''
Select the matching v3.1 projection only for smaller encoders.
只有小型 Encoder 需要對應的 v3.1 ClipProj 投影矩陣。
'''
PROJECTION_FILE = f'mmh3-{ENCODER_SIZE.lower()}-ClipProj-v3.1-mlp.safetensors' if ENCODER_SIZE != '32B' else None

'''
Select the native 32B loader or the small-encoder projection loader.
原生 32B 改用無投影裝載節點，4B／8B 維持 ClipProj。
'''
TEXT_LOADER_NODE = 'ClipProjDeviceLoader' if ENCODER_SIZE == '32B' else 'ClipProjLoader'

'''
Show free temporary disk space for user inspection.
顯示可用暫存磁碟空間，方便自行評估下載容量。
'''
print(f'GPU 0：HyperFlow {QUANT}（{QUANT_SIZES_GB[QUANT]} GB）；GPU 1：{ENCODER_VARIANT}（{ENCODER_GB} GB）；/tmp 剩餘 {shutil.disk_usage("/tmp").free / 1024**3:.1f} GiB')

'''
Download exactly one task-specific GGUF plus its encoder and two VAEs.
只下載所選任務的 FL2VA 或 Ref2VA GGUF，以及 Encoder 與兩個 VAE。
'''
models = [
    (GGUF_REPO, f'{MODEL_FAMILY}_gguf/{DIT_NAME}', 'diffusion_models'),
    (ENCODER_REPO, ENCODER_FILE, 'text_encoders'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_fp16.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
]

'''
Download a ClipProj matrix only for the 4B/8B alternatives.
只有選擇 4B／8B 備案時才下載對應 ClipProj 矩陣。
'''
if PROJECTION_FILE:

    '''
    Include its optional projection checkpoint in the download plan.
    將所需的投影權重加入下載清單。
    '''
    models.append(('NicoLab28/ClipProj-MiniMax-H3', PROJECTION_FILE, 'clip_projections'))

'''
Create the scratch directory before checking its available disk space.
檢查磁碟剩餘空間前，先建立權重暫存目錄。
'''
STORE.mkdir(parents=True, exist_ok=True)

'''
Estimate the additional space needed for all selected model weights (decimal GB).
估算本次所選全部模型權重還需要多少暫存空間（十進位 GB）。
'''
model_sizes = [QUANT_SIZES_GB[QUANT], ENCODER_GB, 5.21, 0.605]

'''
Account for the extra ClipProj projection matrix when used.
使用小型 Encoder 時，額外計入 ClipProj 投影矩陣容量。
'''
if PROJECTION_FILE:

    '''
    Approximate the published residual-MLP projection size for each encoder.
    依 Encoder 大小估算官方 residual-MLP 投影矩陣的容量。
    '''
    model_sizes.append(0.577 if ENCODER_SIZE == '8B' else 0.481)

'''
Include only model files that are not already present in the scratch directory.
只計算尚未存在於暫存資料夾的權重檔案，避免重跑時重複計算。
'''
missing_gb = sum(size for (_, filename, folder), size in zip(models, model_sizes)
                 if not (STORE / folder / filename).is_file())

'''
Read free space on the actual filesystem containing the model directory.
取得真正儲存模型權重的檔案系統剩餘空間。
'''
free_gb = shutil.disk_usage(STORE).free / 1e9

'''
Require a small extra margin for incomplete downloads and metadata.
預留少量空間給未完成下載檔案與必要中繼資料。
'''
assert free_gb >= missing_gb * 1.05, (
    f'/tmp 空間不足：尚需約 {missing_gb:.1f} GB，剩餘 {free_gb:.1f} GB；'
    '請清除暫存模型，或改用較小的 GPU 0／GPU 1 權重。'
)

'''
Show the capacity check before starting any large file transfer.
大型檔案下載前先顯示磁碟容量檢查結果。
'''
print(f'下載容量檢查：尚需約 {missing_gb:.1f} GB；剩餘 {free_gb:.1f} GB。')

'''
Track newly linked files so a running service can rescan dropdowns.
追蹤新增的模型連結，確保已啟動的服務更新選單。
'''
for repo, filename, folder in models:

    '''
    Report each requested model before downloading.
    下載前顯示目前正在處理的模型檔案。
    '''
    print('模型：', Path(filename).name, flush=True)

    '''
    Download the exact file once into scratch storage.
    將指定權重只下載一次到暫存磁碟。
    '''
    source = Path(hf_hub_download(repo_id=repo, filename=filename, local_dir=STORE / folder))

    '''
    Make the model visible to ComfyUI without copying it.
    利用符號連結讓 ComfyUI 辨識模型，避免複製權重。
    '''
    target = COMFY / 'models' / folder / source.name

    '''
    Ensure loader-specific model directories exist.
    確認各個 Loader 的模型資料夾已建立。
    '''
    target.parent.mkdir(parents=True, exist_ok=True)

    '''
    Remove only an outdated symbolic link, not a real model file.
    僅刪除指向其他位置的舊連結，不刪除實體模型檔案。
    '''
    if target.is_symlink() and (not target.exists() or not target.samefile(source)):

        '''
        Replace stale links with the current model file.
        用目前選擇的權重取代失效連結。
        '''
        target.unlink()

    '''
    Create a new link only when needed.
    只有缺少正確連結時才建立新的符號連結。
    '''
    if not target.exists():

        '''
        Link the complete file after Hugging Face confirms download.
        下載完成後建立指向模型權重的符號連結。
        '''
        target.symlink_to(source.resolve())

        '''
        Force server refresh after adding model entries.
        新增模型後要求 ComfyUI 重新掃描模型選單。
        '''
        H3_RESTART_REQUIRED = True

    '''
    Reject unexpected same-name files to avoid silent model substitution.
    遇到同名但內容不同的權重檔案時，避免默默載入錯誤檔案。
    '''
    assert target.samefile(source), f'模型檔案衝突：{target}'

'''
Record both GPU model choices for runtime validation.
記錄 GPU 0／1 的模型選項，供生成前驗證。
'''
H3_MODEL_CHOICE = (MODEL_FAMILY, QUANT, ENCODER_VARIANT)

'''
Confirm the exact downloaded models and the encoder loader.
顯示實際下載的模型與 Encoder Loader。
'''
print(f'✓ 第 2 節完成：{TASK.upper()} / {MODEL_FAMILY} / GPU 0 {QUANT}；GPU 1 {ENCODER_VARIANT}（{TEXT_LOADER_NODE}）；Video/Audio VAE 已就緒。')

## 3. GPU 0／1 與 CPU Offload 設定


In [ ]:
'''
Budget 12.5 GiB on GPU 0 and offload the rest of Q8_0 as needed.
GPU 0 設定 12.5 GiB 預算，其餘 Q8_0 權重視需要卸載。
'''
GPU0_MODEL_GIB = 12.5

'''
Show an 11 GiB GPU 1 monitoring reference, not an allocation limit.
GPU 1 以 11 GiB 作為監測參考值，並非 VRAM 硬上限。
'''
GPU1_MODEL_GIB = 11.0

'''
Allow disk-backed model weights on Kaggle's memory-limited host. Available choices: disk, auto, ram.
在 Kaggle 記憶體有限的環境使用磁碟映射卸載權重。 可選值：disk、auto、ram。
'''
WEIGHTS_IN = 'disk'

'''
Place the native 32B or optional 4B/8B text encoder on GPU 1.
將原生 32B 或選用的 4B／8B 文字編碼器分配到 GPU 1。
'''
TEXT_GPU = 'cuda:1'

'''
Use dynamic text encoder offload to leave GPU 1 compute headroom. Available choices: dynamic, streaming, resident.
文字編碼器使用 dynamic 卸載，保留 GPU 1 推論所需空間。 可選值：dynamic、streaming、resident。
'''
TEXT_MODE = 'dynamic'

'''
Read actual GPU 0 capacity instead of assuming every T4 is identical.
讀取 GPU 0 的實際容量，避免假設所有 T4 完全相同。
'''
gpu0_capacity = torch.cuda.get_device_properties(0).total_memory / 1024**3

'''
Inspect the second GPU independently.
獨立讀取 GPU 1 的 VRAM 容量。
'''
gpu1_capacity = torch.cuda.get_device_properties(1).total_memory / 1024**3

'''
Reject physically impossible budgets and unsupported load modes.
避免設定不可能的 GPU 預算或不支援的載入模式。
'''
assert 0 < GPU0_MODEL_GIB < gpu0_capacity and 0 < GPU1_MODEL_GIB < gpu1_capacity

'''
Allow only options supported by the installed HyperFlow Loader.
只接受 HyperFlow Loader 支援的權重儲存方式。
'''
assert WEIGHTS_IN in ('auto', 'disk', 'ram')

'''
Validate the optional ClipProj offload strategy.
確認 ClipProj 的卸載模式有效。
'''
assert TEXT_MODE in ('dynamic', 'streaming', 'resident')

'''
Reserve GPU 0 VRAM for activation tensors and decoder workloads.
保留 GPU 0 的部分 VRAM 給中間 Tensor 與解碼運算。
'''
GPU0_RESERVE_GIB = round(gpu0_capacity - GPU0_MODEL_GIB, 3)

'''
Choose classic node caching for unchanged model loaders; use none if RAM is insufficient. Available choices: classic, none.
相同模型載入節點使用 classic 快取；RAM 不足時可改成 none。 可選值：classic、none。
'''
NODE_CACHE_MODE = 'classic'

'''
Reject invalid ComfyUI node-cache modes.
避免輸入 ComfyUI 不支援的節點快取模式。
'''
assert NODE_CACHE_MODE in ('classic', 'none')

'''
Detect changes to memory placement across notebook re-runs.
追蹤 Notebook 重跑期間記憶體配置是否更動。
'''
H3_DESIRED_CONFIG = (GPU0_RESERVE_GIB, WEIGHTS_IN, TEXT_GPU, TEXT_MODE, NODE_CACHE_MODE)

'''
Report the target budget without pretending that it is a per-model hard cap.
顯示預期記憶體預算，不宣稱能精確限制單一模型用量。
'''
print(f'✓ 第 3 節完成：GPU 0 目標 {GPU0_MODEL_GIB:.1f} GiB、保留 {GPU0_RESERVE_GIB:.1f} GiB；GPU 1 使用 {TEXT_MODE}，參考值 {GPU1_MODEL_GIB:.1f} GiB（非硬上限）；offload={WEIGHTS_IN}')

## 4. 啟動 ComfyUI 與節點相容性檢查


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import requests
import time

'''
Keep diagnostic logs outside the model files.
將錯誤紀錄與模型權重分開保存。
'''
LOG = ROOT / 'hyperflow_h3_comfyui.log'

'''
Define the private API address for this notebook.
設定 Notebook 專用的本機 API 位址。
'''
BASE = f'http://127.0.0.1:{PORT}'

'''
Test the ComfyUI API without exposing any public tunnel.
不開放公開網址，直接測試本機 ComfyUI API。
'''
def ready():

    '''
    Handle normal startup connection errors without crashing.
    安全處理服務啟動期間尚未開放連線的情況。
    '''
    try:

        '''
        Request server statistics as a readiness signal.
        透過系統狀態 API 判斷服務是否就緒。
        '''
        return requests.get(f'{BASE}/system_stats', timeout=3).ok

    except requests.RequestException:

        '''
        Treat connection failures as not-ready status.
        連線失敗時視為服務尚未就緒。
        '''

        '''
        Let the caller decide whether to retry.
        由呼叫端決定是否繼續等待。
        '''
        return False

'''
Read a node's enumerated dropdown options from the live API.
從服務目前註冊的節點規格讀取下拉式選單。
'''
def choices(node_name, input_name):

    '''
    Inspect the requested node's input declaration.
    讀取指定節點的輸入規格。
    '''
    groups = NODE_INFO[node_name].get('input', {})

    '''
    Search both mandatory and optional input groups.
    同時搜尋必填與選填的輸入項目。
    '''
    for kind in ('required', 'optional'):

        '''
        Retrieve the input specification, if present.
        取得存在的輸入欄位規格。
        '''
        spec = groups.get(kind, {}).get(input_name)

        '''
        Return the dropdown list only when its schema is enumerated.
        只有明確的選單才回傳可選值。
        '''
        if isinstance(spec, (list, tuple)) and spec and isinstance(spec[0], list):

            '''
            Return the exact list from the running node.
            使用目前執行中的節點所提供的選單。
            '''
            return spec[0]

        '''
        Stop when a scalar input exists but is not a dropdown.
        若欄位存在但不是選單，就不自行編造候選值。
        '''
        if spec is not None:

            '''
            Signal that this input has no enumerated choices.
            表示此欄位沒有固定選單。
            '''
            return None

    '''
    Fail early when an upstream plugin renamed the expected field.
    若外掛改名造成預期欄位消失，就提前停止。
    '''
    raise RuntimeError(f'{node_name}.{input_name} 欄位不存在。')

'''
Determine whether the notebook-owned service must restart.
判斷 Notebook 自行啟動的服務是否需要重新啟動。
'''
need_restart = globals().get('H3_RESTART_REQUIRED', False) or globals().get('H3_ACTIVE_CONFIG') != H3_DESIRED_CONFIG

'''
Reuse an existing service only if its model dropdowns are current.
只有模型清單正確時才沿用現有服務。
'''
if ready() and not need_restart:

    '''
    Detect a switched quantization or text encoder before generation.
    在生成前偵測是否切換量化模型或文字編碼器。
    '''
    need_restart = globals().get('H3_ACTIVE_MODEL') != H3_MODEL_CHOICE

'''
Shut down only a child process owned by this notebook kernel.
只允許關閉目前 Notebook Kernel 自己管理的服務程序。
'''
if ready() and need_restart:

    '''
    Avoid killing a model service that belongs to a different session.
    避免終止其他 Notebook 或 Session 的模型服務。
    '''
    assert 'H3_PROC' in globals() and H3_PROC.poll() is None, f'埠 {PORT} 已被其他程序使用；請重啟 Kaggle Session。'

    '''
    Request a graceful process shutdown.
    要求原本的服務正常關閉。
    '''
    H3_PROC.terminate()

    '''
    Release the port before restarting with updated nodes or models.
    在套用新模型或節點前釋放連接埠。
    '''
    try:

        '''
        Bound the shutdown time so a stalled child cannot hang the notebook.
        限制結束程序的等待時間，避免 Notebook 卡住。
        '''
        H3_PROC.wait(timeout=20)

    except subprocess.TimeoutExpired:

        '''
        Recover when the child refuses to exit cleanly.
        處理無法正常結束的子程序。
        '''

        '''
        Terminate only the known notebook-owned process.
        只強制結束目前 Notebook 管理的已知程序。
        '''
        H3_PROC.kill()

        '''
        Wait for the kernel to release process resources.
        等待系統完全回收該程序資源。
        '''
        H3_PROC.wait(timeout=10)

'''
Start the local service if no healthy server is available.
沒有可用服務時才啟動新的本機 ComfyUI。
'''
if not ready():

    '''
    Use disk-backed GGUF streaming and dynamic VRAM management.
    透過磁碟映射卸載 GGUF 權重並管理動態 VRAM。
    '''
    command = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT),
               '--enable-dynamic-vram', '--fast-disk', '--reserve-vram', str(GPU0_RESERVE_GIB),
               '--cache-classic' if NODE_CACHE_MODE == 'classic' else '--cache-none',
               '--disable-pinned-memory']

    '''
    Store complete startup warnings in a diagnostic file.
    將完整的啟動警告保存於診斷檔案。
    '''
    with LOG.open('w', encoding='utf-8') as stream:

        '''
        Run ComfyUI without blocking this notebook cell.
        以非阻塞方式啟動 ComfyUI 服務。
        '''
        H3_PROC = subprocess.Popen(command, cwd=COMFY, stdout=stream, stderr=subprocess.STDOUT)

    '''
    Wait for the real HTTP endpoint rather than a log string.
    等待真正可連線的 HTTP API，而不是僅檢查日誌字串。
    '''
    for attempt in range(180):

        '''
        Surface a crash immediately with its last diagnostic lines.
        服務若已崩潰，立即顯示最後的診斷紀錄。
        '''
        if H3_PROC.poll() is not None:

            '''
            Report the actual ComfyUI initialization failure.
            顯示真實的 ComfyUI 初始化錯誤。
            '''
            raise RuntimeError(LOG.read_text(errors='replace')[-6500:])

        '''
        Proceed once the service responds successfully.
        服務可正常回應後立即繼續。
        '''
        if ready():

            '''
            Exit the startup polling loop.
            結束啟動等待迴圈。
            '''
            break

        '''
        Avoid unnecessary requests while the server initializes.
        服務初始化期間降低查詢頻率。
        '''
        time.sleep(2)

    '''
    Explain any startup timeout using real logs.
    逾時時提供實際日誌，協助定位問題。
    '''
    assert ready(), LOG.read_text(errors='replace')[-6500:]

'''
Record which memory settings the live server is using.
記錄目前服務實際套用的記憶體配置。
'''
H3_ACTIVE_CONFIG = H3_DESIRED_CONFIG

'''
Retrieve registered node types and parameter schemas.
取得已註冊的節點類型與參數規格。
'''
NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()

'''
Require the official HyperFlow nodes rather than the original H3 sampler.
確認使用官方 HyperFlow 節點，而不是舊版 H3 採樣器。
'''
REQUIRED_NODES = (TEXT_LOADER_NODE, 'HyperFlowLoader', 'HyperFlowSigmas',
                  'MiniMaxH3ImageToVideo', 'MiniMaxH3ReferenceToVideo', 'VAELoader', 'BasicGuider', 'KSamplerSelect',
                  'RandomNoise', 'SamplerCustomAdvanced', 'VAEDecode', 'VAEDecodeAudio',
                  'CreateVideo', 'SaveVideo', 'LoadImage')

'''
Collect every missing node before an expensive model load.
在耗費 GPU 時間前一次列出全部缺少的節點。
'''
missing = [name for name in REQUIRED_NODES if name not in NODE_INFO]

'''
Show actionable errors for incompatible or incomplete plugin installs.
外掛版本不相容時顯示可追蹤的具體缺失。
'''
assert not missing, f'ComfyUI 缺少節點 {missing}。請檢查 {LOG}。'

'''
Verify the installed loader actually lists the requested quantization.
確認 HyperFlow Loader 能辨識指定的 GGUF 模型。
'''
assert DIT_NAME in (choices('HyperFlowLoader', 'unet_name') or []), f'HyperFlow Loader 未辨識模型：{DIT_NAME}'

'''
Verify the native or projected text encoder is visible to its loader.
確認選定的原生或投影 Encoder 能被對應節點辨識。
'''
assert Path(ENCODER_FILE).name in (choices(TEXT_LOADER_NODE, 'clip_name') or []), '找不到 Qwen3-VL Encoder。'

'''
Validate the projection only when selecting a smaller encoder.
只有選擇小型 Encoder 時才確認 ClipProj 投影矩陣。
'''
if PROJECTION_FILE:

    '''
    Require the matching v3.1 projection before generation.
    在生成前確認相容的 v3.1 投影矩陣已存在。
    '''
    assert PROJECTION_FILE in (choices(TEXT_LOADER_NODE, 'projection') or []), '找不到 ClipProj v3.1 投影矩陣。'

'''
Verify that GPU 1 is offered by the active encoder loader.
確認所選 Encoder Loader 支援 GPU 1。
'''
assert TEXT_GPU in (choices(TEXT_LOADER_NODE, 'device') or []), f'{TEXT_LOADER_NODE} 不支援 {TEXT_GPU}。'

'''
Check dynamic/streaming/resident mode against the installed node.
依已安裝節點確認 dynamic／streaming／resident 選項。
'''
assert TEXT_MODE in (choices(TEXT_LOADER_NODE, 'mode') or []), 'Encoder Loader 不支援指定的 Offload 模式。'

'''
Require the native MiniMax text-encoder architecture when using 32B.
選擇原生 32B 時確認節點有 MiniMax 的文字編碼器型別。
'''
if ENCODER_SIZE == '32B':

    '''
    Avoid loading the 32B weights under an unrelated encoder type.
    避免將 32B 權重當成其他架構載入。
    '''
    assert 'minimax' in (choices(TEXT_LOADER_NODE, 'type') or []), '缺少 MiniMax 原生 Encoder 型別。'

'''
Validate the exact video and audio decoder weights.
確認影片與音訊 VAE 權重都已被辨識。
'''
for name in ('minimax_h3_video_vae_fp16.safetensors', 'minimax_h3_audio_vae_fp32.safetensors'):

    '''
    Stop when a required VAE is absent from the running model registry.
    若必要 VAE 沒有出現在目前模型清單，立即停止。
    '''
    assert name in (choices('VAELoader', 'vae_name') or []), f'找不到 VAE：{name}'

'''
Confirm that only genuine HyperFlow schedules will be used.
確認採樣改用 HyperFlow 的原生 sigma 排程。
'''
assert 'steps' in NODE_INFO['HyperFlowSigmas']['input'].get('optional', {}), 'HyperFlow Sigmas 缺少 steps 欄位。'

'''
Verify the canonical Euler sampler remains available.
確認官方 HyperFlow 所需的 Euler Sampler 已存在。
'''
assert 'euler' in (choices('KSamplerSelect', 'sampler_name') or []), 'Sampler 不支援 euler。'

'''
Record the model pair validated by the running server.
記錄目前 ComfyUI 已確認的模型與 Encoder 組合。
'''
H3_ACTIVE_MODEL = H3_MODEL_CHOICE

'''
Clear the restart flag after all node and weight checks pass.
全部檢查通過後清除重新啟動標記。
'''
H3_RESTART_REQUIRED = False

'''
Confirm the server is ready for generation.
確認 HyperFlow 服務已可接受生成工作。
'''
print(f'✓ 第 4 節完成：GPU 0 {QUANT} / GPU 1 {ENCODER_VARIANT} / Disk Offload。')

## 5. T2V：Prompt 與生成參數


In [ ]:
'''
Describe the text-to-video scene and synchronized sound.
設定文生影片的場景與同步音效描述。
'''
T2V_PROMPT = '''
A cinematic close-up of a red panda in a misty forest. The red panda slowly turns its head.
Natural fur details, gentle camera movement, soft morning light. Sound: birds chirping and light wind.
'''.strip()

'''
Configure the output dimensions for text-to-video.
設定文生影片的畫面解析度。
'''
T2V_WIDTH, T2V_HEIGHT = 384, 224

'''
Configure the target duration in seconds.
設定文生影片的目標秒數。
'''
T2V_SECONDS = 10

'''
Select the official HyperFlow 4-, 6-, or 8-step schedule.
選擇官方 HyperFlow 的 4、6 或 8 steps 排程。
'''
T2V_STEPS = 6

'''
Choose the HyperFlow quantization independently for this task. Available choices: Q3_K_M, Q4_K_S, Q4_K_M, Q5_K_M, Q6_K, Q8_0.
獨立指定任務使用的 HyperFlow 模型量化。 可選值：Q3_K_M、Q4_K_S、Q4_K_M、Q5_K_M、Q6_K、Q8_0。
'''
T2V_QUANT = 'Q8_0'

'''
Encoder sizes and formats; B means billion parameters. Select one complete option; native 32B needs no ClipProj. Independently choose GPU 1's real encoder format and model size.
Encoder 模型大小與格式；B 代表十億個參數（32B=320億、8B=80億、4B=40億）。 Setting / 設定值  | Model / 模型                 | Precision / 權重精度 32B_NVFP4       | Qwen3-VL-32B（原生）          | NVFP4（4-bit 浮點） 32B_INT8        | Qwen3-VL-32B（原生）          | INT8 ConvRot（8-bit 整數） 32B_BF16        | Qwen3-VL-32B（原生）          | BF16（16-bit 浮點） 8B_NVFP4        | Qwen3-VL-8B + ClipProj      | NVFP4（4-bit 浮點） 8B_W4A8         | Qwen3-VL-8B + ClipProj      | W4A8（4-bit 權重／8-bit activation） 8B_INT8         | Qwen3-VL-8B + ClipProj      | INT8 ConvRot（8-bit 整數） 8B_FP8          | Qwen3-VL-8B + ClipProj      | FP8（8-bit 浮點） 8B_BF16         | Qwen3-VL-8B + ClipProj      | BF16（16-bit 浮點） 4B_FP8          | Qwen3-VL-4B + ClipProj      | FP8（8-bit 浮點） 4B_BF16         | Qwen3-VL-4B + ClipProj      | BF16（16-bit 浮點） 請填寫完整選項；原生 32B 不需 ClipProj，8B／4B 則需要。 獨立選擇 GPU 1 真正提供的 Encoder 權重格式與模型大小。
'''
T2V_GPU1_ENCODER = '8B_FP8'

'''
Set this task’s seed; use -1 to randomize each generation.
設定此任務的種子；-1 代表每次生成都重新抽取。
'''
T2V_SEED = 42

'''
Set the T2V output filename without overwriting earlier runs.
設定文生影片輸出檔名；若同名會自動編號。
'''
T2V_OUTPUT = 'hyperflow_h3_flash_t2v.mp4'

'''
Preserve configurations for every task independently.
分開保存各項任務的設定，避免不同 Prompt 互相覆寫。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register this task's complete generation settings.
登記文生影片的完整生成設定。
'''
TASK_CONFIGS['t2v'] = dict(prompt=T2V_PROMPT, width=T2V_WIDTH, height=T2V_HEIGHT,
                           seconds=T2V_SECONDS, steps=T2V_STEPS, seed=T2V_SEED,
                           encoder_variant=T2V_GPU1_ENCODER, quant=T2V_QUANT,
                           output_name=T2V_OUTPUT, first_image='', last_image='')

'''
Provide explicit feedback after configuring the task.
完成任務設定後顯示明確提示。
'''
print(f'✓ 第 5 節完成：T2V | {T2V_WIDTH}×{T2V_HEIGHT} | {T2V_SECONDS} 秒 | {T2V_STEPS} steps | {T2V_QUANT} | GPU 1 {T2V_GPU1_ENCODER} | {T2V_OUTPUT}')

## 6. I2V：Prompt、圖片與生成參數


In [ ]:
'''
Describe how the uploaded first frame should animate.
描述上傳的第一張圖片要如何轉化為動態影片。
'''
I2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()

'''
Select the initial image from the Kaggle filesystem.
指定 Kaggle 本地的起始圖片路徑。
'''
I2V_FIRST_IMAGE = '/kaggle/working/first_frame.png'

'''
Configure output dimensions for image-to-video.
設定圖生影片的解析度。
'''
I2V_WIDTH, I2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定圖生影片的目標秒數。
'''
I2V_SECONDS = 10

'''
Use the HyperFlow steps supported by the official scheduler.
使用官方 HyperFlow 排程器支援的步數。
'''
I2V_STEPS = 6

'''
Choose the HyperFlow quantization independently for this task. Available choices: Q3_K_M, Q4_K_S, Q4_K_M, Q5_K_M, Q6_K, Q8_0.
獨立指定任務使用的 HyperFlow 模型量化。 可選值：Q3_K_M、Q4_K_S、Q4_K_M、Q5_K_M、Q6_K、Q8_0。
'''
I2V_QUANT = 'Q8_0'

'''
Encoder sizes and formats; B means billion parameters. Select one complete option; native 32B needs no ClipProj. Independently choose GPU 1's real encoder format and model size.
Encoder 模型大小與格式；B 代表十億個參數（32B=320億、8B=80億、4B=40億）。 Setting / 設定值  | Model / 模型                 | Precision / 權重精度 32B_NVFP4       | Qwen3-VL-32B（原生）          | NVFP4（4-bit 浮點） 32B_INT8        | Qwen3-VL-32B（原生）          | INT8 ConvRot（8-bit 整數） 32B_BF16        | Qwen3-VL-32B（原生）          | BF16（16-bit 浮點） 8B_NVFP4        | Qwen3-VL-8B + ClipProj      | NVFP4（4-bit 浮點） 8B_W4A8         | Qwen3-VL-8B + ClipProj      | W4A8（4-bit 權重／8-bit activation） 8B_INT8         | Qwen3-VL-8B + ClipProj      | INT8 ConvRot（8-bit 整數） 8B_FP8          | Qwen3-VL-8B + ClipProj      | FP8（8-bit 浮點） 8B_BF16         | Qwen3-VL-8B + ClipProj      | BF16（16-bit 浮點） 4B_FP8          | Qwen3-VL-4B + ClipProj      | FP8（8-bit 浮點） 4B_BF16         | Qwen3-VL-4B + ClipProj      | BF16（16-bit 浮點） 請填寫完整選項；原生 32B 不需 ClipProj，8B／4B 則需要。 獨立選擇 GPU 1 真正提供的 Encoder 權重格式與模型大小。
'''
I2V_GPU1_ENCODER = '8B_FP8'

'''
Set this task’s seed; use -1 to randomize each generation.
設定此任務的種子；-1 代表每次生成都重新抽取。
'''
I2V_SEED = 42

'''
Set an independent image-to-video output filename.
指定圖生影片自己的輸出檔名。
'''
I2V_OUTPUT = 'hyperflow_h3_flash_i2v.mp4'

'''
Reuse the task configuration registry without resetting other tasks.
沿用任務設定清單，不重置其他任務。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register the first-frame conditioning input and task parameters.
登記起始影格及圖生影片參數。
'''
TASK_CONFIGS['i2v'] = dict(prompt=I2V_PROMPT, width=I2V_WIDTH, height=I2V_HEIGHT,
                           seconds=I2V_SECONDS, steps=I2V_STEPS, seed=I2V_SEED,
                           encoder_variant=I2V_GPU1_ENCODER, quant=I2V_QUANT,
                           output_name=I2V_OUTPUT, first_image=I2V_FIRST_IMAGE, last_image='')

'''
Confirm the configuration cell has run without attempting to open the image yet.
顯示設定完成；實際圖片檔案會在生成時再驗證。
'''
print(f'✓ 第 6 節完成：I2V | {I2V_WIDTH}×{I2V_HEIGHT} | {I2V_SECONDS} 秒 | {I2V_QUANT} | GPU 1 {I2V_GPU1_ENCODER} | 圖片：{I2V_FIRST_IMAGE}')

## 7. FLF2V：Prompt、首尾圖片與生成參數


In [ ]:
'''
Describe a video that connects its first and last reference frames.
描述從第一張圖片過渡到最後一張圖片的影片內容。
'''
FLF2V_PROMPT = '''
Create a smooth cinematic transition from the first reference frame to the last reference frame.
Maintain consistent subject details and lighting, with natural intermediate motion and synchronized ambient sound.
'''.strip()

'''
Choose the first image in the start/end-frame sequence.
設定首尾影格模式的起始圖片。
'''
FLF2V_FIRST_IMAGE = '/kaggle/working/first_frame.png'

'''
Choose the last image in the start/end-frame sequence.
設定首尾影格模式的結束圖片。
'''
FLF2V_LAST_IMAGE = '/kaggle/working/last_frame.png'

'''
Set output dimensions for first-and-last-frame generation.
設定首尾影格模式的輸出解析度。
'''
FLF2V_WIDTH, FLF2V_HEIGHT = 384, 224

'''
Set the duration for the start/end-frame transition.
設定首尾影格過渡的目標秒數。
'''
FLF2V_SECONDS = 10

'''
Select 4, 6, or 8 HyperFlow sampling steps.
設定 4、6 或 8 步的 HyperFlow 採樣。
'''
FLF2V_STEPS = 6

'''
Choose the HyperFlow quantization independently for this task. Available choices: Q3_K_M, Q4_K_S, Q4_K_M, Q5_K_M, Q6_K, Q8_0.
獨立指定任務使用的 HyperFlow 模型量化。 可選值：Q3_K_M、Q4_K_S、Q4_K_M、Q5_K_M、Q6_K、Q8_0。
'''
FLF2V_QUANT = 'Q8_0'

'''
Encoder sizes and formats; B means billion parameters. Select one complete option; native 32B needs no ClipProj. Independently choose GPU 1's real encoder format and model size.
Encoder 模型大小與格式；B 代表十億個參數（32B=320億、8B=80億、4B=40億）。 Setting / 設定值  | Model / 模型                 | Precision / 權重精度 32B_NVFP4       | Qwen3-VL-32B（原生）          | NVFP4（4-bit 浮點） 32B_INT8        | Qwen3-VL-32B（原生）          | INT8 ConvRot（8-bit 整數） 32B_BF16        | Qwen3-VL-32B（原生）          | BF16（16-bit 浮點） 8B_NVFP4        | Qwen3-VL-8B + ClipProj      | NVFP4（4-bit 浮點） 8B_W4A8         | Qwen3-VL-8B + ClipProj      | W4A8（4-bit 權重／8-bit activation） 8B_INT8         | Qwen3-VL-8B + ClipProj      | INT8 ConvRot（8-bit 整數） 8B_FP8          | Qwen3-VL-8B + ClipProj      | FP8（8-bit 浮點） 8B_BF16         | Qwen3-VL-8B + ClipProj      | BF16（16-bit 浮點） 4B_FP8          | Qwen3-VL-4B + ClipProj      | FP8（8-bit 浮點） 4B_BF16         | Qwen3-VL-4B + ClipProj      | BF16（16-bit 浮點） 請填寫完整選項；原生 32B 不需 ClipProj，8B／4B 則需要。 獨立選擇 GPU 1 真正提供的 Encoder 權重格式與模型大小。
'''
FLF2V_GPU1_ENCODER = '8B_FP8'

'''
Set this task’s seed; use -1 to randomize each generation.
設定此任務的種子；-1 代表每次生成都重新抽取。
'''
FLF2V_SEED = 42

'''
Save output under a dedicated filename.
為首尾影格模式指定獨立輸出檔名。
'''
FLF2V_OUTPUT = 'hyperflow_h3_flash_flf2v.mp4'

'''
Keep previously configured modes unchanged.
保留先前其它生成模式的設定。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register both conditioning images and all parameters.
登記兩張參考圖片與全部生成參數。
'''
TASK_CONFIGS['flf2v'] = dict(prompt=FLF2V_PROMPT, width=FLF2V_WIDTH, height=FLF2V_HEIGHT,
                             seconds=FLF2V_SECONDS, steps=FLF2V_STEPS, seed=FLF2V_SEED,
                             encoder_variant=FLF2V_GPU1_ENCODER, quant=FLF2V_QUANT,
                             output_name=FLF2V_OUTPUT, first_image=FLF2V_FIRST_IMAGE,
                             last_image=FLF2V_LAST_IMAGE)

'''
Confirm the configuration has been stored without running generation.
確認首尾影格設定已儲存，這一格不會執行生成。
'''
print(f'✓ 第 7 節完成：FLF2V | {FLF2V_WIDTH}×{FLF2V_HEIGHT} | {FLF2V_SECONDS} 秒 | {FLF2V_QUANT} | GPU 1 {FLF2V_GPU1_ENCODER} | 參考圖 2 張')

## 8. R2V：Prompt、參考圖片與生成參數


In [ ]:
'''
Describe the primary picture; the prompt stays valid when selecting only one image.
描述第一張參考圖；即使只選一張圖片，Prompt 仍能正常使用。
'''
R2V_PROMPT = '''
<Picture 1> defines the main subject's appearance.
Create a cinematic scene with natural motion and ambient sound.
Keep the main subject consistent, and use additional pictures as visual references when supplied.
'''.strip()

'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import io
import tempfile
import shutil
from html import escape

'''
Use exact three-digit reference filenames in a dedicated directory.
在獨立資料夾中，使用從 001 開始的三位數圖片檔名。
'''
R2V_REF_DIR = ROOT / 'hyperflow_h3_r2v_references'

'''
Accept only the image formats supported by the video graph.
僅允許 Workflow 支援的圖片格式。
'''
R2V_FORMATS = {'.png': 'PNG', '.jpg': 'JPEG', '.jpeg': 'JPEG', '.webp': 'WEBP'}

'''
Preserve the most recently confirmed batch across cell reruns in this Kaggle session.
在同一個 Kaggle Session 中重跑 Cell 時，沿用最後一次成功選取的圖片組合。
'''
R2V_REF_DIR.mkdir(parents=True, exist_ok=True)

'''
Read only previously numbered image files, sorted by their three-digit codes.
只載入先前按三位數編號的圖片，並依照編號排序。
'''
R2V_ACTIVE_FILES = sorted(p for p in R2V_REF_DIR.iterdir()
                          if p.is_file() and p.suffix.lower() in R2V_FORMATS
                          and len(p.stem) == 3 and p.stem.isdigit())

'''
Refuse gaps or duplicates to prevent mapping a different file to <Picture N>.
不允許缺號或重複編號，避免 <Picture N> 指向錯誤圖片。
'''
assert len(R2V_ACTIVE_FILES) <= 9 and all(
    path.stem == f'{index:03d}' for index, path in enumerate(R2V_ACTIVE_FILES, 1)
), 'R2V 圖片資料夾編號不連續；請重新選取整組圖片。'

'''
Keep the public reference list synchronized with the files actually on disk.
讓公開的參考圖片清單與磁碟上的實際檔案保持一致。
'''
R2V_REF_IMAGES = [str(path) for path in R2V_ACTIVE_FILES]

'''
Select reference processing: match reduces tokens; max preserves more details but needs more VRAM.
參考圖尺寸：match 較省記憶體；max 保留較多細節但更耗 VRAM。 Options / 可選值: 'match'（依輸出面積縮圖）, 'max'（參考圖短邊最多 2048px）。
'''
R2V_REF_IMAGE_SIZE = 'match'

'''
Set the output canvas; each dimension must be a positive multiple of 32.
設定輸出畫布；寬高皆須為正數且為 32 的倍數。
'''
R2V_WIDTH, R2V_HEIGHT = 384, 224

'''
Set the approximate video duration in seconds.
設定目標影片秒數，實際影格數會依 H3 規則對齊。
'''
R2V_SECONDS = 10

'''
Use the official HyperFlow 4-, 6-, or 8-step sigma schedule.
採樣步數只支援官方 HyperFlow 的 4、6、8 steps。
'''
R2V_STEPS = 6

'''
Choose the independent Ref2VA GGUF precision, not an FL2VA checkpoint.
選用獨立 Ref2VA GGUF 權重，不能拿 FL2VA 權重取代。 Options / 可選值: 'Q3_K_M' (~7.4 GB), 'Q4_K_S' (~8.1 GB), 'Q4_K_M' (~9.2 GB), 'Q5_K_M' (~10.5 GB), 'Q6_K' (~11.8 GB), 'Q8_0' (~15.2 GB;需 offload)。
'''
R2V_QUANT = 'Q4_K_M'

'''
Encoder sizes and precisions; B means one billion parameters. Prefer native 32B for picture references; it needs GPU 1 offload on T4.
Encoder 大小與精度；B 代表十億個參數，32B=320 億、8B=80 億、4B=40 億。 Setting / 設定值  | Model / 模型                 | Precision / 權重精度 32B_NVFP4       | 原生 Qwen3-VL-32B            | NVFP4（4-bit 浮點） 32B_INT8        | 原生 Qwen3-VL-32B            | INT8 ConvRot（8-bit 整數） 32B_BF16        | 原生 Qwen3-VL-32B            | BF16（16-bit 浮點） 8B_NVFP4        | Qwen3-VL-8B + ClipProj      | NVFP4（4-bit 浮點） 8B_W4A8         | Qwen3-VL-8B + ClipProj      | W4A8（4-bit 權重／8-bit activation） 8B_INT8         | Qwen3-VL-8B + ClipProj      | INT8 ConvRot（8-bit 整數） 8B_FP8          | Qwen3-VL-8B + ClipProj      | FP8（8-bit 浮點） 8B_BF16         | Qwen3-VL-8B + ClipProj      | BF16（16-bit 浮點） 4B_FP8          | Qwen3-VL-4B + ClipProj      | FP8（8-bit 浮點） 4B_BF16         | Qwen3-VL-4B + ClipProj      | BF16（16-bit 浮點） 多圖參考預設使用原生 32B；在 T4 上須透過 GPU 1 卸載管理。 8B／4B + ClipProj 可節省資源，但多圖參考的影像條件品質尚需實測。
'''
R2V_GPU1_ENCODER = '32B_NVFP4'

'''
Set this task’s seed; use -1 to randomize each generation.
設定此任務的種子；-1 代表每次生成都重新抽取。
'''
R2V_SEED = 42

'''
Name the output of this independent reference-to-video task.
設定多圖參考模式獨立的輸出檔名。
'''
R2V_OUTPUT = 'hyperflow_h3_flash_r2v.mp4'

'''
Retain the other three task configurations in the same notebook session.
保留同一個 Notebook Session 內其他三種生成模式的參數。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register all reference-to-video settings without altering the other modes.
登記 R2V 的參考圖片與模型設定，不更動其他模式。
'''
TASK_CONFIGS['r2v'] = dict(prompt=R2V_PROMPT, width=R2V_WIDTH, height=R2V_HEIGHT,
                           seconds=R2V_SECONDS, steps=R2V_STEPS, seed=R2V_SEED,
                           encoder_variant=R2V_GPU1_ENCODER, quant=R2V_QUANT,
                           output_name=R2V_OUTPUT, first_image='', last_image='',
                           ref_images=R2V_REF_IMAGES, ref_image_size=R2V_REF_IMAGE_SIZE)

'''
Prepare a status area that lists the exact 001/002/... to <Picture N> mapping.
建立狀態區，顯示 001／002 等檔案與 <Picture N> 的對應順序。
'''
R2V_UPLOAD_STATUS = widgets.HTML()

'''
Keep the active file picker replaceable so the next selection cannot append old uploads.
讓選檔元件可重新建立，確保下次多選不會累積上一批檔案。
'''
R2V_UPLOAD_PANEL = widgets.VBox()

'''
Describe the automatic replacement policy above the file picker.
在選檔元件上方說明：每次多選都會完整取代舊圖片。
'''
R2V_UPLOAD_HELP = widgets.HTML(value=(
    '<b>R2V 參考圖片（1～9 張）</b>：一次選取所有圖片；每次重新選取會完整取代上一批，'
    '從 001 重新編號，並刪除不再使用的舊圖片。'
))

'''
Apply one complete browser selection as a new image set, never merging it with an earlier batch.
將瀏覽器本次選取視為完整的新圖片組合，絕不與上一批合併。
'''
def on_r2v_upload(change):

    '''
    Update the global image list when a valid new batch is committed.
    新圖片批次確認有效後，更新全域參考圖片清單。
    '''
    global R2V_REF_IMAGES

    '''
    The upload widget can emit an empty event while resetting; ignore it.
    上傳元件重設時可能產生空事件，此時不修改現有圖片。
    '''
    uploaded = change['new']

    '''
    A cancelled or empty file picker must not destroy a previous selection.
    使用者取消選檔或沒有選取圖片時，不刪除原本的圖片。
    '''
    if not uploaded:

        '''
        Keep the last successful batch unchanged.
        保留上一次成功套用的圖片組合。
        '''
        return

    '''
    Do not modify any existing file until the new images pass validation.
    新圖片全部通過驗證前，不修改原先任何檔案。
    '''
    staging = None

    '''
    Show a concise success or failure message after validation.
    驗證完成後顯示簡潔的成功或失敗訊息。
    '''
    message = ''

    '''
    Try to stage and atomically replace the entire selected batch.
    嘗試建立暫存圖片組合，再以整批替換方式更新正式資料夾。
    '''
    try:

        '''
        MiniMax H3 R2V accepts a maximum of nine reference images.
        MiniMax H3 R2V 最多只能使用九張參考圖片。
        '''
        assert 1 <= len(uploaded) <= 9, '請一次選取 1～9 張圖片。'

        '''
        Store the batch in a temporary sibling directory, away from active files.
        先將本次選取放入同一磁碟的暫存目錄，不覆蓋有效圖片。
        '''
        staging = Path(tempfile.mkdtemp(prefix='.hyperflow_r2v_new_', dir=ROOT))

        '''
        Show the user which original name maps to each reference code.
        顯示原始檔名與新的三位數代號之間的對照關係。
        '''
        mapping = []

        '''
        Keep the browser's selected file order for <Picture 1> onward.
        按照瀏覽器回傳的選取順序，對應 <Picture 1> 等圖片標籤。
        '''
        for index, item in enumerate(uploaded, 1):

            '''
            Ignore any client-side directory information in the original filename.
            忽略原始檔名中的任何客戶端路徑資訊。
            '''
            original = item['name'].replace('\\', '/').split('/')[-1]

            '''
            Preserve the true extension without keeping the original filename.
            保留圖片副檔名，但不沿用原始檔名。
            '''
            extension = Path(original).suffix.lower()

            '''
            Allow PNG, JPEG, and WebP images only.
            只接受 PNG、JPEG 與 WebP 格式。
            '''
            assert extension in R2V_FORMATS, f'不支援的圖片格式：{original}'

            '''
            Read the binary upload content sent through the browser widget.
            讀取瀏覽器元件送到 Python Kernel 的圖片位元資料。
            '''
            content = bytes(item['content'])

            '''
            Reject an empty upload before opening the image decoder.
            在開啟圖片解碼器前先拒絕空白檔案。
            '''
            assert content, f'圖片內容為空：{original}'

            '''
            Validate the actual file format, not only its extension.
            檢查圖片的真實格式，不只依賴副檔名。
            '''
            with Image.open(io.BytesIO(content)) as picture:

                '''
                Reject files disguised as supported formats.
                拒絕副檔名與圖片實際格式不符的檔案。
                '''
                assert picture.format == R2V_FORMATS[extension], f'圖片內容格式不符：{original}'

                '''
                Verify image structure without recompressing the source.
                驗證圖片結構，並保留原始畫質不重新壓縮。
                '''
                picture.verify()

            '''
            Allocate a stable, zero-padded code starting from 001.
            從 001 開始，為本批圖片產生固定長度三位數代號。
            '''
            filename = f'{index:03d}{extension}'

            '''
            Save only verified bytes inside the isolated staging directory.
            只將通過驗證的圖片寫入獨立暫存目錄。
            '''
            (staging / filename).write_bytes(content)

            '''
            Record the input order for the user-facing mapping.
            記錄本次上傳順序，供畫面顯示對照表。
            '''
            mapping.append(f'&lt;Picture {index}&gt; = {escape(filename)} ← {escape(original)}')

        '''
        Reserve a unique backup directory for the previous successful batch.
        替上一批有效圖片保留一個獨立備份目錄。
        '''
        backup = ROOT / f'.hyperflow_r2v_old_{__import__("uuid").uuid4().hex}'

        '''
        Move the previous selection out of the active directory in one operation.
        一次將上一批圖片資料夾移至備份位置。
        '''
        R2V_REF_DIR.rename(backup)

        '''
        Restore the previous selection if the new directory cannot be installed.
        如果無法啟用新圖片資料夾，就將上一批圖片完整復原。
        '''
        try:

            '''
            Publish the verified batch as the sole active image directory.
            將已驗證的新批次整批設為唯一有效的圖片目錄。
            '''
            staging.rename(R2V_REF_DIR)

        except Exception:

            '''
            Restore the old selection if publication unexpectedly fails.
            如果替換過程發生例外，復原原先圖片資料夾。
            '''

            '''
            Recover all previous images without leaving a mixed batch.
            復原原本整組圖片，不讓新舊圖片混在一起。
            '''
            backup.rename(R2V_REF_DIR)

            '''
            Propagate the original problem to the status area.
            將真正的錯誤原因交由狀態區顯示。
            '''
            raise

        '''
        Delete all retired filenames, including extra files from a longer prior batch.
        刪除舊批次全部檔案，包括前次較多而本次不再使用的編號。
        '''
        shutil.rmtree(backup, ignore_errors=True)

        '''
        Read only the currently numbered files in exact Picture order.
        只讀取現在有效的圖片，並依 Picture 順序排列。
        '''
        R2V_REF_IMAGES = [str(path) for path in sorted(R2V_REF_DIR.iterdir()) if path.is_file()]

        '''
        Synchronize the generation cell immediately without requiring a parameter rerun.
        立即同步第 9 節的生成設定，不必重新執行參數 Cell。
        '''
        TASK_CONFIGS['r2v']['ref_images'] = list(R2V_REF_IMAGES)

        '''
        Show exactly which files are now active and the reference-label mapping.
        顯示目前唯一有效的圖片及其參考標籤對照。
        '''
        message = '<b>已套用最新 ' + str(len(uploaded)) + ' 張圖片：</b><br>' + '<br>'.join(mapping)

    except Exception as error:

        '''
        Preserve the earlier successful images when validation or disk writes fail.
        若圖片驗證或寫入失敗，維持前一次成功的圖片組合。
        '''

        '''
        Report a clear failure without claiming the current set was replaced.
        明確顯示失敗原因，避免誤以為原有圖片已被替換。
        '''
        message = '<b>未更新參考圖片：</b>' + escape(str(error))

    finally:

        '''
        Always clean up any unused staging directory and reset the browser picker.
        無論成功或失敗，均清除暫存目錄並重新建立瀏覽器選檔元件。
        '''

        '''
        Remove a staging directory that was not published.
        刪除未成功套用的暫存圖片目錄。
        '''
        if staging is not None and staging.exists():

            '''
            Avoid leaving temporary uploaded files behind.
            避免留下沒有使用的暫存圖片。
            '''
            shutil.rmtree(staging, ignore_errors=True)

        '''
        Display the result and current Picture mapping.
        顯示本次更新結果與圖片對應編號。
        '''
        R2V_UPLOAD_STATUS.value = message

        '''
        Replace the FileUpload widget to guarantee independent next-file selection.
        重建 FileUpload 元件，確保下次選圖不累積此次檔案。
        '''
        R2V_UPLOAD_PANEL.children = (new_r2v_uploader(), R2V_UPLOAD_STATUS)

'''
Create a new browser file picker and subscribe to its batch-upload event.
建立新的瀏覽器選檔元件，並監聽每批圖片上傳完成事件。
'''
def new_r2v_uploader():

    '''
    Offer one browser picker for multiple image files.
    提供可一次選取多張圖片的瀏覽器元件。
    '''
    uploader = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=True,
                                   description='多選參考圖片')

    '''
    Trigger replacement only after the current batch has reached the Python kernel.
    等本次圖片完整送達 Python Kernel 後才執行整批替換。
    '''
    uploader.observe(on_r2v_upload, names='value')

    '''
    Let the caller install the fresh picker above the status area.
    將新的選檔元件交由介面安裝在狀態區上方。
    '''
    return uploader

'''
Show the currently active selection before any new upload is made.
在重新選圖前顯示目前已生效的圖片清單。
'''
R2V_UPLOAD_STATUS.value = ('目前圖片：' + (', '.join(Path(path).name for path in R2V_REF_IMAGES)
                                          if R2V_REF_IMAGES else '尚未選擇'))

'''
Display the active file picker and status in one compact web UI.
將選檔元件及狀態訊息組合成簡潔的網頁介面。
'''
R2V_UPLOAD_PANEL.children = (new_r2v_uploader(), R2V_UPLOAD_STATUS)

'''
Render the upload widget directly in this R2V settings cell.
直接在 R2V 參數 Cell 顯示圖片上傳元件。
'''
display(widgets.VBox([R2V_UPLOAD_HELP, R2V_UPLOAD_PANEL]))

'''
Confirm task settings and explain when model changes require a restart.
確認參考圖與模型設定，並提醒只有更換模型時才需重新下載與啟動。
'''
print(f'✓ 第 8 節完成：R2V | {R2V_WIDTH}×{R2V_HEIGHT} | {len(R2V_REF_IMAGES)} 張參考圖 | {R2V_QUANT} | GPU 1 {R2V_GPU1_ENCODER}。圖片重新選取後可直接執行第 9 節。')


## 9. 生成影片（T2V／I2V／FLF2V／R2V）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import json
import shutil
import uuid
import math
import websocket
from tqdm.auto import tqdm

'''
Use the task explicitly selected in the model-download section.
沿用模型下載章節指定的 TASK，確保 GPU 0 下載正確架構。 Options / 可選值: 't2v', 'i2v', 'flf2v', 'r2v'。
'''
TASK = globals().get('TASK', 't2v')

'''
Read the selected task's previously executed configuration cell.
讀取所選任務已執行過的設定 Cell。
'''
assert TASK in globals().get('TASK_CONFIGS', {}), f'請先執行 {TASK.upper()} 的參數設定 Cell。'

'''
Resolve the selected task's independent settings.
取得所選任務的獨立參數。
'''
config = TASK_CONFIGS[TASK]

'''
Share sampling, decoding and video output logic across four task modes.
四種生成模式共用採樣、解碼及影片輸出流程。
'''
MODE = TASK

'''
Use the prompt stored in the selected task cell.
使用目前任務設定 Cell 裡的 Prompt。
'''
PROMPT = config['prompt']

'''
Use the selected canvas settings.
套用所選任務的輸出尺寸。
'''
WIDTH, HEIGHT = config['width'], config['height']

'''
Use the task-specific duration and sampling count.
套用所選任務的秒數與採樣步數。
'''
SECONDS, STEPS = config['seconds'], config['steps']

'''
Apply the task-specific seed and output filename.
套用所選任務的種子與輸出檔名。
'''
SEED, OUTPUT_NAME = config['seed'], config['output_name']

'''
Resolve task-specific reference images without mixing modes.
依據所選任務使用對應參考圖片，避免不同模式混用。
'''
FIRST_IMAGE, LAST_IMAGE = config['first_image'], config['last_image']

'''
Read only the most recently confirmed picture selection for R2V.
R2V 只讀取最後一次成功多選後的圖片組合。
'''
REF_IMAGES = list(config.get('ref_images', []))

'''
Choose reference resizing mode (match or max) for R2V.
R2V 參考圖片使用 match 或 max 尺寸處理模式。
'''
REF_IMAGE_SIZE = config.get('ref_image_size', 'match')

'''
Read GPU 1's exact text encoder variant from this task's configuration.
從本次任務設定讀取 GPU 1 的文字編碼器版本。
'''
ENCODER_VARIANT = config['encoder_variant']

'''
Derive the encoder architecture needed by the selected loader.
取得 Loader 所需的文字編碼器架構大小。
'''
ENCODER_SIZE = ENCODER_VARIANT.split('_', 1)[0]

'''
Apply the selected HyperFlow GGUF quantization for model consistency checks.
使用此任務指定的 HyperFlow GGUF 量化格式，以確保模型檢查一致。
'''
QUANT = config['quant']

'''
Reject unsupported quantization names before attempting generation.
生成前先攔截不支援的量化格式名稱。
'''
assert QUANT in QUANT_SIZES_GB and ENCODER_VARIANT in ENCODERS, f'不支援的 GPU 模型組合：{QUANT}/{ENCODER_VARIANT}'

'''
Use the official HyperFlow sigma schedules only.
僅使用官方 HyperFlow 的 4、6、8 步 Sigma 排程。
'''
assert STEPS in (4, 6, 8), 'HyperFlow STEPS 僅支援 4、6、8。'

'''
Ensure edited GPU 0 memory budgets have been applied to the running service.
確認調整過的 GPU 0 記憶體預算已套用到目前服務。
'''
assert globals().get('H3_ACTIVE_CONFIG') == H3_DESIRED_CONFIG, 'GPU 0 預算已變更，請重新執行第 4 節啟動 ComfyUI。'

'''
Prevent generation until the chosen quantization was downloaded and registered.
切換量化格式後，必須先完成下載與 ComfyUI 模型註冊。
'''
assert globals().get('H3_MODEL_CHOICE') == (MODEL_FAMILY, QUANT, ENCODER_VARIANT) and globals().get('H3_ACTIVE_MODEL') == (MODEL_FAMILY, QUANT, ENCODER_VARIANT), (
    f'目前選擇 GPU 0 {QUANT}／GPU 1 {ENCODER_VARIANT}，但尚未完成下載或服務預檢。請重跑第 2 節與第 4 節。'
)

'''
Ensure the workflow file and encoder filenames match the selected weights.
確認 Workflow 的 GPU 0／1 檔名都對應目前選擇的權重。
'''
assert MODEL_FAMILY == ('ref2va' if TASK == 'r2v' else 'fl2va'), '目前 GPU 0 模型架構與任務不一致，請重跑第 2、4 節。'

'''
Validate the selected diffusion architecture and quantization against the downloaded file.
同時檢查模型架構與精度，避免載入上一個任務的 GGUF。
'''
assert DIT_NAME == f'minimax_h3_{MODEL_FAMILY}_hyperflow_flash_14b-{QUANT}.gguf', 'GPU 0 模型檔名不一致，請重跑第 2 節。'

'''
Reject a stale GPU 1 encoder that belongs to an earlier selection.
避免切換 GPU 1 模型後仍使用前一次載入的 Encoder。
'''
assert Path(ENCODER_FILE).name == ENCODERS[ENCODER_VARIANT][1], 'GPU 1 模型檔名不一致，請重跑第 2 節。'

'''
Avoid showing a stale result from an earlier successful run after a failure.
避免本次生成失敗後，最後一格仍誤顯示舊影片。
'''
LAST_OUTPUT = None

'''
Confirm the existing local service is still running.
確認先前啟動的本地推論服務仍正常運作。
'''
assert ready(), 'ComfyUI 已中斷，請先重新執行第 3 節。'

'''
Validate the generation mode, prompt, sampling steps, and positive duration.
檢查生成模式、Prompt、採樣步數及秒數是否有效。
'''
assert MODE in ('t2v', 'i2v', 'flf2v', 'r2v') and PROMPT.strip() and STEPS > 0 and math.isfinite(SECONDS) and SECONDS > 0, (
    '請確認生成模式、Prompt、STEPS 與 SECONDS 設定正確；SECONDS 必須為正數。'
)

'''
Require positive dimensions aligned to 32, without an artificial resolution limit.
寬高需為正數且符合 32 的倍數，不另外限制解析度或長寬比。
'''
assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 32 == 0 and HEIGHT % 32 == 0, (
    'WIDTH 與 HEIGHT 必須為正數且是 32 的倍數。'
)

'''
Calculate a valid number of H3 frames (17n + 5).
計算符合 H3 規則的影格數（17n + 5）。
'''
FRAMES = max(22, 17 * math.ceil((SECONDS * 24 - 5) / 17) + 5)

'''
Respect the native MiniMax H3 node's current 3600-frame schema limit.
遵守官方 MiniMax H3 節點目前最多 3600 frames 的規格。
'''
assert FRAMES <= 3600, f'影格數 {FRAMES} 超出官方 H3 節點上限 3600，請縮短影片秒數。'

'''
Generate a fresh 32-bit seed only when -1 is requested.
只有設定 -1 時才建立新的 32-bit 隨機種子。
'''
import secrets

'''
Resolve one seed per run without overwriting the notebook setting.
每次生成決定一個種子，但不覆寫 Notebook 的原始設定。
'''
RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)

'''
Require a supported seed before building the inference workflow.
在建立推論工作流程前，先檢查種子範圍。
'''
assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'

'''
Show the actual seed so a random run can be reproduced later.
顯示實際使用的種子，方便之後重現隨機生成結果。
'''
print(f'本次生成 seed：{RUN_SEED}')

'''
Build a unique ComfyUI output prefix from the user filename.
根據指定檔名建立不重複的 ComfyUI 輸出前綴。
'''
output_stem = Path(OUTPUT_NAME).stem

'''
Make each run independent even when settings are unchanged.
即使設定相同，也為每次生成保留獨立檔案。
'''
run_name = f'{output_stem}_{QUANT}_{uuid.uuid4().hex[:8]}'

'''
Configure the native 32B loader or a smaller ClipProj encoder.
設定原生 32B Encoder，或需要投影矩陣的小型 Encoder。
'''
text_inputs = {'clip_name': Path(ENCODER_FILE).name,
               'type': 'minimax' if ENCODER_SIZE == '32B' else 'auto',
               'device': TEXT_GPU, 'mode': TEXT_MODE}

'''
Add the learned projection only when a 4B/8B encoder is selected.
只有使用 4B／8B Encoder 時才接入學習式投影矩陣。
'''
if PROJECTION_FILE:

    '''
    Keep the matched projection in the encoder loader's inputs.
    將相符的投影矩陣加入 Encoder Loader 的參數。
    '''
    text_inputs['projection'] = PROJECTION_FILE

'''
Build the official HyperFlow GGUF API graph with task-specific prompts.
以官方 HyperFlow GGUF 節點建立任務專用 Workflow。
'''
graph = {
    '1': {'class_type': TEXT_LOADER_NODE, 'inputs': text_inputs},
    '2': {'class_type': 'HyperFlowLoader', 'inputs': {'unet_name': DIT_NAME, 'weights_in': WEIGHTS_IN}},
    '4': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_video_vae_fp16.safetensors'}},
    '5': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_audio_vae_fp32.safetensors'}},
    '6': {'class_type': ('MiniMaxH3ReferenceToVideo' if MODE == 'r2v' else 'MiniMaxH3ImageToVideo'), 'inputs': {
        'clip': ['1', 0], 'vae': ['4', 0], 'prompt': PROMPT,
        'width': WIDTH, 'height': HEIGHT, 'length': FRAMES}},
    '7': {'class_type': 'BasicGuider', 'inputs': {
        'model': ['2', 0], 'conditioning': ['6', 0]}},
    '8': {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler'}},
    '9': {'class_type': 'HyperFlowSigmas', 'inputs': {'model': ['2', 0], 'steps': STEPS}},
    '10': {'class_type': 'RandomNoise', 'inputs': {'noise_seed': RUN_SEED}},
    '11': {'class_type': 'SamplerCustomAdvanced', 'inputs': {
        'noise': ['10', 0], 'guider': ['7', 0],
        'sampler': ['8', 0], 'sigmas': ['9', 0], 'latent_image': ['6', 1]}},
    '12': {'class_type': 'VAEDecode', 'inputs': {'samples': ['11', 0], 'vae': ['4', 0]}},
    '13': {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['11', 0], 'vae': ['5', 0]}},
    '14': {'class_type': 'CreateVideo', 'inputs': {
        'images': ['12', 0], 'audio': ['13', 0], 'fps': 24.0}},
    '15': {'class_type': 'SaveVideo', 'inputs': {
        'video': ['14', 0], 'filename_prefix': run_name, 'format': 'mp4'}},
}

'''
Supply the dedicated Ref2VA conditioner with the Audio VAE and reference sizing mode.
R2V 使用 Ref2VA 專用條件節點，並接入 Audio VAE 與參考圖尺寸模式。
'''
if MODE == 'r2v':

    '''
    Make sure the user explicitly provided between one and nine image paths.
    確保使用者明確提供 1～9 張參考圖片路徑。
    '''
    assert isinstance(REF_IMAGES, (list, tuple)) and 1 <= len(REF_IMAGES) <= 9, 'R2V_REF_IMAGES 必須包含 1～9 張圖片。'

    '''
    Reject missing or superseded references before loading any image into ComfyUI.
    載入圖片到 ComfyUI 前，拒絕不存在或已被新批次取代的圖片。
    '''
    assert REF_IMAGES == [str(path) for path in sorted(R2V_REF_DIR.iterdir()) if path.is_file()], (
        'R2V 圖片清單與目前選取的圖片不一致；請重新執行第 8 節。'
    )

    '''
    Restrict the reference resize mode to the two official choices.
    參考圖尺寸選項只能填寫官方支援的 match 或 max。
    '''
    assert REF_IMAGE_SIZE in ('match', 'max'), 'R2V_REF_IMAGE_SIZE 只能是 match 或 max。'

    '''
    Match image numbering in the prompt to the 1-based reference ordering.
    核對 Prompt 的圖片編號是否與參考圖片順序一致（從 1 起算）。
    '''
    import re

    '''
    Read all explicit Picture labels in the user-supplied prompt.
    取得 Prompt 中全部 <Picture N> 標籤。
    '''
    mentioned = {int(number) for number in re.findall(r'<Picture\s+(\d+)>', PROMPT, re.IGNORECASE)}

    '''
    Avoid silently referencing an image that was not provided.
    避免 Prompt 指向不存在的參考圖片。
    '''
    assert mentioned and mentioned <= set(range(1, len(REF_IMAGES) + 1)), 'Prompt 必須使用有效的 <Picture 1> 等標籤。'

    '''
    Connect the actual audio VAE used by the reference conditioner.
    將實際使用的 Audio VAE 連入 R2V 條件節點。
    '''
    graph['6']['inputs']['audio_vae'] = ['5', 0]

    '''
    Apply the documented image-sizing method for every reference.
    對所有參考圖套用官方定義的縮放模式。
    '''
    graph['6']['inputs']['ref_image_size'] = REF_IMAGE_SIZE

    '''
    Preserve the exact user-defined reference ordering.
    保留使用者指定的參考圖片順序。
    '''
    for index, image_path in enumerate(REF_IMAGES):

        '''
        Check the uploaded image before copying it into ComfyUI.
        複製到 ComfyUI 前先確認圖片存在且有效。
        '''
        source = Path(image_path)

        '''
        Reject missing, empty or unsupported image input files.
        拒絕不存在、空白或格式不支援的圖片檔案。
        '''
        assert source.is_file() and source.stat().st_size > 0 and source.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp'), f'無效的 R2V 參考圖片：{source}'

        '''
        Reserve a unique node ID for this reference image.
        每張參考圖使用獨立的節點編號。
        '''
        node_id = str(20 + index)

        '''
        Store each reference under a unique ComfyUI input filename.
        每張參考圖使用獨立檔名放入 ComfyUI input 目錄。
        '''
        destination = COMFY / 'input' / f'r2v_{index + 1}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'

        '''
        Create the input directory if it does not exist.
        視需要建立 ComfyUI input 資料夾。
        '''
        destination.parent.mkdir(parents=True, exist_ok=True)

        '''
        Copy the reference image without modifying its content.
        保留參考圖內容並複製到輸入資料夾。
        '''
        shutil.copy2(source, destination)

        '''
        Register this image with the official LoadImage node.
        透過官方 LoadImage 節點載入圖片。
        '''
        graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}

        '''
        Use zero-based autogrow sockets; text Picture labels use one-based indexing.
        API 使用從 0 起算的 autogrow 欄位；Prompt 標籤則從 1 起算。
        '''
        graph['6']['inputs'][f'ref_images.ref_image_{index}'] = [node_id, 0]

'''
Attach a verified local image to an H3 image-conditioning socket.
將確認存在的本地圖片接到 H3 圖片條件輸入。
'''
for mode, image_path, node_id, input_name in [
    ('i2v', FIRST_IMAGE, '16', 'first_frame'),
    ('flf2v', FIRST_IMAGE, '16', 'first_frame'),
    ('flf2v', LAST_IMAGE, '17', 'last_frame'),
]:

    '''
    Only attach images required by the active mode.
    只接入目前模式需要的圖片。
    '''
    if MODE == mode:

        '''
        Require an existing image file for the selected mode.
        確認所選模式的圖片檔案確實存在。
        '''
        source = Path(image_path)

        '''
        Stop before generation if a reference file is missing.
        若圖片不存在，在生成前立即停止。
        '''
        assert source.is_file(), f'找不到參考圖片：{source}'

        '''
        Copy the input to the ComfyUI image directory.
        將圖片複製到 ComfyUI 輸入目錄。
        '''
        destination = COMFY / 'input' / f'{node_id}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'

        '''
        Create the target input folder when needed.
        視需要建立圖片輸入資料夾。
        '''
        destination.parent.mkdir(parents=True, exist_ok=True)

        '''
        Preserve the reference image without changing it.
        保留原始圖片內容，不進行修改。
        '''
        shutil.copy2(source, destination)

        '''
        Register the image loader in the API graph.
        將圖片載入節點加入 API Workflow。
        '''
        graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}

        '''
        Connect the loaded image to H3 conditioning.
        將已載入圖片連到 H3 條件輸入。
        '''
        graph['6']['inputs'][input_name] = [node_id, 0]

'''
Re-read relevant dropdowns because model lists or user parameters may change after startup.
在提交前重新確認下拉選單，避免啟動後變更模型或設定造成不相容。
'''
NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()

'''
Validate the user's GPU 1 choice against the actual encoder node.
依照當前編碼器節點的選項檢查 GPU 1 設定。
'''
assert TEXT_GPU in (choices(TEXT_LOADER_NODE, 'device') or []), 'Encoder Loader 不支援 GPU 1。'

'''
Validate the selected encoder offload mode.
驗證選擇的文字編碼器卸載模式。
'''
assert TEXT_MODE in (choices(TEXT_LOADER_NODE, 'mode') or []), 'Encoder Loader 不支援指定 Offload 模式。'

'''
Verify that the official HyperFlow Euler sampler is available.
確認官方 HyperFlow Euler Sampler 可用。
'''
assert 'euler' in (choices('KSamplerSelect', 'sampler_name') or []), 'HyperFlow 必須使用 Euler Sampler。'

'''
Ensure the published HyperFlow Sigma scheduler is present.
確認官方 HyperFlow Sigma 排程器已註冊。
'''
assert 'HyperFlowSigmas' in NODE_INFO, '缺少 HyperFlow Sigma 節點。'

'''
Confirm optional first and last frame sockets exist for the selected image modes.
在使用圖片生成影片時，確認節點確實支援首尾影格輸入。
'''
if MODE in ('i2v', 'flf2v'):

    '''
    Read the available optional conditioning inputs from the running node definition.
    讀取執行中影格條件節點的選填輸入規格。
    '''
    frame_inputs = NODE_INFO['MiniMaxH3ImageToVideo'].get('input', {}).get('optional', {})

    '''
    Require the first frame, plus the last frame only in FLF2V mode.
    圖生影片必須支援第一格；首尾影格模式還需要最後一格。
    '''
    expected = ('first_frame', 'last_frame') if MODE == 'flf2v' else ('first_frame',)

    '''
    Reject an incompatible core node implementation before generation.
    如果核心節點不支援這些欄位，在生成之前先停止。
    '''
    assert all(field in frame_inputs for field in expected), f'MiniMaxH3ImageToVideo 缺少 {expected} 輸入。'

'''
Validate that all graph dependency links target existing nodes.
驗證所有 Workflow 連線的來源節點確實存在。
'''
for node_id, node in graph.items():

    '''
    Examine each declared input for possible node-reference links.
    檢查節點輸入中是否包含連到其它節點的參照。
    '''
    for value in node['inputs'].values():

        '''
        Only two-element lists are ComfyUI graph connections.
        只有雙元素清單才可能代表 ComfyUI 節點連線。
        '''
        if isinstance(value, list) and len(value) == 2:

            '''
            Stop immediately if a linked source node is missing.
            若連結到不存在的來源節點，立即顯示具體錯誤。
            '''
            assert str(value[0]) in graph, f'節點 {node_id} 連結到不存在的節點 {value[0]}。'

'''
Compare each workflow input against the actual node input definitions.
依照執行中 ComfyUI 的節點規格檢查每個 Workflow 參數。
'''
for node_id, node in graph.items():

    '''
    Read this node's current ComfyUI schema and accepted input names.
    讀取目前節點允許的輸入欄位名稱。
    '''
    schema = NODE_INFO[node['class_type']].get('input', {})

    '''
    Include required, optional and system-provided hidden inputs.
    將必填、選填與系統隱藏欄位全部納入檢查。
    '''
    accepted = set().union(*(set(schema.get(group, {})) for group in ('required', 'optional', 'hidden')))

    '''
    Expand the official R2V autogrow image sockets into their API dotted names.
    將官方 R2V autogrow 圖片欄位展開為 API 使用的點號名稱。
    '''
    if node['class_type'] == 'MiniMaxH3ReferenceToVideo':

        '''
        Confirm the running node has the reference-images autogrow input.
        確認目前執行中的節點支援參考圖片 autogrow 輸入。
        '''
        assert 'ref_images' in accepted, 'MiniMaxH3ReferenceToVideo 不支援 ref_images；請更新 ComfyUI。'

        '''
        Include only the numbered sockets actually used in this request.
        只加入本次真正使用的參考圖片欄位，不接受任意未知參數。
        '''
        accepted.update(f'ref_images.ref_image_{index}' for index in range(len(REF_IMAGES)))

    '''
    Find parameters removed or renamed by a later ComfyUI revision.
    找出可能被新版 ComfyUI 移除或改名的參數。
    '''
    unknown = set(node['inputs']) - accepted

    '''
    Fail early if a workflow input is no longer supported.
    當 Workflow 參數不再被節點支援時，立即停止並指出名稱。
    '''
    assert not unknown, f"節點 {node_id} ({node['class_type']}) 不支援的輸入：{sorted(unknown)}"

'''
Print the selected task and workload before expensive sampling.
在開始耗時生成前顯示本次任務與工作量。
'''
print(f'{MODE.upper()} | {WIDTH}×{HEIGHT} | {FRAMES} frames | {STEPS} steps | GPU 0 {QUANT} | GPU 1 {ENCODER_VARIANT} | HyperFlow', flush=True)

'''
Assign a unique WebSocket client identity to this generation.
為此次生成建立獨立的 WebSocket 客戶端識別碼。
'''
client_id = uuid.uuid4().hex

'''
Track WebSocket availability separately from workflow success.
個別追蹤 WebSocket 可用性，不影響實際生成流程。
'''
ws = None

'''
Connect before queueing to avoid missing early progress messages.
在送出任務前先連線，避免漏掉初期進度事件。
'''
try:

    '''
    Connect to the documented ComfyUI progress event stream.
    連接 ComfyUI 官方 WebSocket 進度事件串流。
    '''
    ws = websocket.create_connection(f'ws://127.0.0.1:{PORT}/ws?clientId={client_id}', timeout=5)

    '''
    Limit individual reads so failure and completion are checked regularly.
    限制單次等待時間，定期檢查錯誤與任務完成狀態。
    '''
    ws.settimeout(2)

except (OSError, websocket.WebSocketException):

    '''
    Continue inference even when live progress is unsupported.
    即使即時進度連線失敗，也不影響模型推論。
    '''

    '''
    Mark event updates as unavailable and retain an elapsed-time display.
    停用事件更新，改成單一進度列顯示經過時間。
    '''
    ws = None

    '''
    Explain why no step percentage will be shown.
    說明為什麼目前不會出現步數百分比。
    '''
    print('即時步數事件不可用；將顯示階段與經過時間。', flush=True)

'''
Send the workflow to the private local ComfyUI service.
將任務送交本機 ComfyUI 服務。
'''
response = requests.post(f'{BASE}/prompt', json={'prompt': graph, 'client_id': client_id}, timeout=120)

'''
Report invalid workflows before creating a misleading progress bar.
工作流程驗證失敗時先回報，不顯示誤導性的進度條。
'''
if not response.ok:

    '''
    Close an unused WebSocket before raising the server-side error.
    回報錯誤前先關閉不再使用的 WebSocket。
    '''
    if ws is not None:

        '''
        Release the client connection.
        釋放 WebSocket 連線。
        '''
        ws.close()

    '''
    Preserve ComfyUI's validation details.
    保留 ComfyUI 提供的參數驗證錯誤資訊。
    '''
    raise RuntimeError(response.text[:5000])

'''
Remember the prompt ID so progress messages belong to this exact task.
保留任務 ID，避免將其它生成工作的進度混入。
'''
prompt_id = response.json()['prompt_id']

'''
Start timing the actual queued job.
開始計算已提交生成工作的耗時。
'''
start = time.monotonic()

'''
Create a single notebook progress bar with real step events only.
建立單一 Notebook 進度條，僅使用真實採樣步數更新。
'''
pbar = tqdm(total=STEPS, desc='排隊／載入模型', unit='step', leave=True)

'''
Preserve a final history record for successful completion checking.
保留最終生成紀錄，用於確認任務完成。
'''
item = None

'''
Avoid repeatedly polling history when the WebSocket reports frequent events.
WebSocket 事件頻繁時避免過度查詢生成紀錄。
'''
last_history_check = 0.0

'''
Define truthful node-stage names for messages without step progress.
將節點 ID 對應到真實執行階段，沒有步數時仍可辨識狀態。
'''
phases = {'1': '文字編碼器', '2': '載入 HyperFlow GGUF', '6': '建立影片條件',
          '11': '採樣中', '12': '影片解碼', '13': '音訊解碼',
          '14': '合成影片', '15': '儲存影片'}

'''
Wait until ComfyUI reports a final history record for this task.
持續等待 ComfyUI 回傳此任務的最終執行紀錄。
'''
try:

    '''
    Keep processing until success, an explicit error, or timeout.
    持續處理直到成功、明確錯誤或逾時。
    '''
    while True:

        '''
        Measure real wall-clock time instead of estimating model progress.
        計算真實經過時間，不將時間誤當模型百分比。
        '''
        elapsed = time.monotonic() - start

        '''
        Receive a WebSocket message if this ComfyUI version supports it.
        如果目前版本支援 WebSocket，讀取即時進度事件。
        '''
        message = None

        '''
        Read events only while the client connection remains healthy.
        只有 WebSocket 正常連線時才讀取事件。
        '''
        if ws is not None:

            '''
            Keep timeout exceptions from stopping generation.
            WebSocket 讀取逾時不應中斷生成。
            '''
            try:

                '''
                Receive a progress event or stage transition.
                接收採樣進度或執行階段變更。
                '''
                message = ws.recv()

            except websocket.WebSocketTimeoutException:

                '''
                A quiet model-loading phase is normal and not a failure.
                模型載入期間沒有事件屬正常情況，不代表失敗。
                '''

                '''
                Continue checking job history after a read timeout.
                讀取逾時後繼續檢查任務紀錄。
                '''
                pass

            except (OSError, websocket.WebSocketException):

                '''
                Recover gracefully when the event channel becomes disconnected.
                進度通道中斷時回退到紀錄輪詢，不影響生成。
                '''

                '''
                Release the unusable client connection.
                關閉失效的 WebSocket 連線。
                '''
                ws.close()

                '''
                Disable further reads from the disconnected socket.
                停用已中斷的進度連線。
                '''
                ws = None

        '''
        Only JSON text frames contain progress data; preview images are binary.
        只有 JSON 文字事件含進度資料；預覽圖片為二進位資料。
        '''
        if isinstance(message, str):

            '''
            Parse a ComfyUI WebSocket event.
            解析 ComfyUI WebSocket 事件。
            '''
            event = json.loads(message)

            '''
            Read the event payload without assuming optional keys exist.
            讀取事件資料，並避免假設選填欄位一定存在。
            '''
            data = event.get('data') or {}

            '''
            Reject progress from another client or generation.
            忽略非本次任務的進度，避免步數混亂。
            '''
            if data.get('prompt_id') == prompt_id:

                '''
                Track which part of the workflow is executing.
                追蹤目前執行中的 workflow 節點。
                '''
                if event.get('type') == 'executing' and data.get('node'):

                    '''
                    Show the real stage rather than inventing a percentage.
                    顯示實際節點階段，不虛構完成百分比。
                    '''
                    pbar.set_description(phases.get(str(data['node']), '執行中'), refresh=False)

                '''
                Use official step updates when the sampler reports them.
                採樣器回報步數時，使用官方即時進度。
                '''
                if event.get('type') in ('progress', 'progress_state'):

                    '''
                    Classic events store step counts directly in the payload.
                    傳統進度事件直接在資料中提供步數。
                    '''
                    progress = data

                    '''
                    Extended events report per-node progress within a nodes object.
                    新版事件使用 nodes 字典提供各節點步數。
                    '''
                    if event.get('type') == 'progress_state':

                        '''
                        Read only this workflow's sampling node.
                        只讀取此 workflow 的採樣節點進度。
                        '''
                        progress = data.get('nodes', {}).get('11', {})

                    '''
                    Ignore progress emitted by loading or decoding nodes.
                    忽略非採樣節點的進度，以免誤顯示百分比。
                    '''
                    if str(progress.get('node', progress.get('node_id'))) == '11':

                        '''
                        Read the actual completed and total sampling steps.
                        取得 ComfyUI 回報的已完成與總採樣步數。
                        '''
                        value, maximum = progress.get('value'), progress.get('max')

                        '''
                        Validate numeric counters before modifying the display.
                        先確認進度欄位為有效數值，再更新畫面。
                        '''
                        if isinstance(value, (int, float)) and isinstance(maximum, (int, float)) and maximum > 0:

                            '''
                            Follow the sampler's actual total if it differs from the setting.
                            如果採樣器回報不同步數，優先使用實際總數。
                            '''
                            pbar.total = int(maximum)

                            '''
                            Update the bar to the real step count, never a time-based estimate.
                            依據真實步數更新進度，不使用耗時推估。
                            '''
                            pbar.n = max(0, min(int(value), pbar.total))

                            '''
                            Identify the phase accurately.
                            正確標示為採樣階段。
                            '''
                            pbar.set_description('採樣中', refresh=False)

        '''
        Check the persistent history regardless of WebSocket availability.
        不論 WebSocket 是否正常，都定期檢查 ComfyUI 任務紀錄。
        '''
        if elapsed - last_history_check >= 4:

            '''
            Record the most recent history query to avoid duplicate requests.
            更新上次查詢時間，避免頻繁重複送出請求。
            '''
            last_history_check = elapsed

            '''
            Show recent logs if the service died under memory pressure.
            如果服務因記憶體壓力中斷，顯示最近的錯誤 Log。
            '''
            if not ready():

                '''
                Preserve recent server logs for actionable diagnostics.
                保留最近服務紀錄供後續除錯。
                '''
                raise RuntimeError('ComfyUI 服務中斷：\n' + LOG.read_text(errors='replace')[-4500:])

            '''
            Read this prompt's final history record.
            讀取本次生成工作的最終執行紀錄。
            '''
            item = requests.get(f'{BASE}/history/{prompt_id}', timeout=60).json().get(prompt_id)

            '''
            Stop polling once the job has a final record.
            任務產生最終紀錄後停止輪詢。
            '''
            if item:

                '''
                Extract node-specific execution errors before claiming success.
                在宣告成功前確認各節點是否有執行錯誤。
                '''
                errors = [m for m in item.get('status', {}).get('messages', []) if m[0] == 'execution_error']

                '''
                Report the precise node failure when one is present.
                如果某個節點失敗，回報其完整錯誤資訊。
                '''
                if errors:

                    '''
                    Surface the error from the last failed node.
                    顯示最後一個失敗節點的錯誤。
                    '''
                    raise RuntimeError(str(errors[-1][1])[:5000])

                '''
                Complete only after ComfyUI history confirms success.
                只有 ComfyUI 紀錄確認完成才結束等待。
                '''
                break

        '''
        Update one stable progress row with elapsed time and no log flooding.
        在同一條進度列更新經過時間，避免持續洗版。
        '''
        pbar.set_postfix_str(f'{elapsed / 60:.1f} 分鐘', refresh=False)

        '''
        Refresh the existing row in-place.
        只刷新原進度列，不新增大量訊息行。
        '''
        pbar.refresh()

        '''
        Avoid tight polling if the WebSocket connection is unavailable.
        WebSocket 不可用時稍作等待，避免過密輪詢。
        '''
        if ws is None:

            '''
            Yield CPU time while generation runs in the server.
            讓伺服器推論期間不持續佔用 Notebook CPU。
            '''
            time.sleep(1)

    '''
    Mark the sampler display finished only after the full job succeeded.
    只有整個工作流程成功後才將進度條標示為完成。
    '''
    pbar.n = pbar.total

    '''
    Label the status as completed.
    將顯示狀態設為完成。
    '''
    pbar.set_description('完成', refresh=False)

    '''
    Refresh the final elapsed-time counter.
    更新任務完成時的總耗時。
    '''
    pbar.set_postfix_str(f'{(time.monotonic() - start) / 60:.1f} 分鐘', refresh=False)

    '''
    Show the final progress state.
    顯示最終完成狀態。
    '''
    pbar.refresh()

finally:

    '''
    Always free the live progress widgets and connection after this job.
    無論任務成功或失敗，都釋放進度列與 WebSocket 連線。
    '''

    '''
    Stop updating the notebook progress widget.
    停止更新 Notebook 的進度顯示。
    '''
    pbar.close()

    '''
    Close the WebSocket when it exists.
    若 WebSocket 仍存在，關閉它以釋放資源。
    '''
    if ws is not None:

        '''
        Release the local event subscription.
        釋放本地進度事件連線。
        '''
        ws.close()

'''
Locate the generated video in ComfyUI output directories.
從 ComfyUI 輸出資料夾找出本次生成影片。
'''
files = [p for p in (COMFY / 'output').rglob('*') if p.is_file() and p.name.startswith(run_name)]

'''
Keep only playable video formats.
只保留可播放的影片格式。
'''
files = [p for p in files if p.suffix.lower() in ('.mp4', '.webm', '.mov')]

'''
Refuse to claim success when no video was saved.
沒有生成影片時，不誤判為完成。
'''
assert files, f'工作流程完成但未找到影片：{item.get("outputs", {})}'

'''
Use the most recently modified matching video file.
選取符合本次輸出名稱的最新影片。
'''
source_video = max(files, key=lambda path: path.stat().st_mtime)

'''
Save the user-visible output in Kaggle working storage.
將成果儲存至 Kaggle working 目錄。
'''
OUTPUT_DIR = ROOT / 'hyperflow_h3_outputs'

'''
Ensure the output directory exists.
確保輸出資料夾存在。
'''
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

'''
Preserve the actual video container extension.
保留生成影片實際使用的副檔名。
'''
output_base = OUTPUT_DIR / f'{Path(OUTPUT_NAME).stem}_{QUANT}{source_video.suffix}'

'''
Generate an unused path instead of overwriting prior outputs.
產生未使用的檔名，避免覆蓋先前影片。
'''
output = output_base

'''
Number duplicate filenames deterministically.
依序為重複檔名編號。
'''
for index in range(2, 1000):

    '''
    Stop once an available filename is found.
    找到可使用的檔名後停止。
    '''
    if not output.exists():

        '''
        Preserve the current unique output path.
        保留目前可用的輸出檔名。
        '''
        break

    '''
    Allocate the next numbered filename.
    配置下一個編號檔名。
    '''
    output = output_base.with_name(f'{output_base.stem}_{index:03d}{output_base.suffix}')

'''
Copy the completed video to the user's output directory.
將影片複製到使用者的輸出目錄。
'''
shutil.copy2(source_video, output)

'''
Report the result, elapsed time, and inference cache behavior.
顯示檔案、耗時與模型快取狀態。
'''
print(f'完成：{output}｜耗時 {(time.monotonic() - start) / 60:.1f} 分鐘')

'''
Save the latest successful result for the final preview/download cell.
將本次成功生成的影片提供給最後一格預覽與下載。
'''
LAST_OUTPUT = output

'''
Confirm that the video is ready without embedding it in the generation cell.
確認影片已輸出，生成 Cell 本身不顯示預覽。
'''
print('✓ 第 9 節完成：影片已生成；請執行第 10 節預覽與下載。')

## 10. 影片預覽、下載與 GPU 使用量


In [ ]:
'''
Display the most recently generated video and its download link.
顯示最近一次成功生成影片的預覽與下載連結。
'''
from IPython.display import Video, FileLink, display

'''
Require a real output from the preceding generation cell.
確認前一個生成 Cell 已成功輸出影片。
'''
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 8 節生成影片。'

'''
Embed the generated video preview only in this final cell.
只在最後一格嵌入影片預覽，避免生成 Cell 輸出太多內容。
'''
display(Video(str(LAST_OUTPUT), embed=True, width=480))

'''
Provide a clickable Kaggle file download link.
顯示 Kaggle 影片檔案下載連結。
'''
display(FileLink(str(LAST_OUTPUT)))

'''
Check both GPUs without unloading any cached model weights.
在不卸載模型快取的情況下查詢兩張 GPU 的顯示記憶體。
'''
status = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.used,memory.total',
     '--format=csv,noheader,nounits'], capture_output=True, text=True, check=True
)

'''
Show the actual readings without claiming they are exact model-memory allocations.
顯示實際使用量，不將其誤當作單一模型的精確權重用量。
'''
print('GPU 記憶體使用量（MiB）：')

'''
Display both devices in a readable form.
逐行顯示兩張 GPU 的使用狀態。
'''
print(status.stdout.strip())

'''
Compare GPU readings with advisory thresholds without claiming hard caps.
將兩張 GPU 的用量與警示值比較，不把它們誤認成精確硬上限。
'''
for line in status.stdout.splitlines():

    '''
    Split GPU usage fields reported by nvidia-smi.
    拆解 nvidia-smi 回傳的 GPU 使用量欄位。
    '''
    fields = [part.strip() for part in line.split(',')]

    '''
    Check either GPU and a well-formed memory reading.
    確認資料屬於兩張 GPU 之一，且包含有效用量數值。
    '''
    if len(fields) == 4 and fields[0] in ('0', '1') and fields[2].isdigit():

        '''
        Convert MiB readings to the same GiB unit as the user setting.
        將 MiB 轉換成與使用者設定相同的 GiB 單位。
        '''
        used_gib = int(fields[2]) / 1024

        '''
        Compare each device with its relevant configured target.
        將每張 GPU 的使用量與各自設定的目標進行比較。
        '''
        limit_gib = GPU0_MODEL_GIB if fields[0] == '0' else GPU1_MODEL_GIB

        '''
        Report an advisory warning if the whole device exceeds its target.
        實際整張 GPU 用量超過目標時顯示提醒，不將目標誤認為硬上限。
        '''
        if used_gib > limit_gib:

            '''
            Explain why the threshold cannot force CPU offload.
            說明警示門檻本身無法強制卸載至 CPU。
            '''
            print(f'提醒：GPU {fields[0]} 使用 {used_gib:.2f} GiB，超過目標／監控值 {limit_gib:.2f} GiB；非硬上限。')

'''
Confirm the notebook's preview and download step finished.
明確顯示預覽與下載步驟已完成。
'''
print(f'✓ 第 10 節完成：影片預覽與下載連結已顯示；{LAST_OUTPUT.name}')


## 11. 擷取最後影格（選用）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import json
from IPython.display import FileLink, display
from PIL import Image
import shutil

'''
Stop with a clear message if the Kaggle environment lacks FFmpeg.
若 Kaggle 環境未安裝 FFmpeg，立即顯示清楚的錯誤訊息。
'''
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), '找不到 ffmpeg 或 ffprobe；請先安裝 FFmpeg。'

'''
Ensure a generated MP4 exists before extracting its last frame.
確認前面已成功生成 MP4，再進行最後影格擷取。
'''
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 8 節生成影片，並確認第 9 節可正常預覽。'

'''
Build a PNG path next to the generated video.
在生成影片旁建立最後影格 PNG 檔案路徑。
'''
LAST_FRAME_PATH = LAST_OUTPUT.with_name(LAST_OUTPUT.stem + '_last_frame.png')

'''
Read the frame count from the saved MP4.
從已儲存的 MP4 讀取影格總數。
'''
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
     '-show_entries', 'stream=nb_read_frames', '-of', 'json', str(LAST_OUTPUT)],
    capture_output=True, text=True, check=True,
)

'''
Convert the frame count to a zero-based index.
將影格總數轉成從零開始的最後影格索引。
'''
last_index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1

'''
Save the exact last frame as a PNG for the next I2V clip.
將最後一個影格儲存成 PNG，供下一段 I2V 使用。
'''
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(LAST_OUTPUT),
     '-vf', rf'select=eq(n\,{last_index})', '-frames:v', '1', str(LAST_FRAME_PATH)],
    check=True,
)

'''
Preview and link the extracted image.
顯示擷取圖片並提供下載連結。
'''
display(Image.open(LAST_FRAME_PATH))
display(FileLink(str(LAST_FRAME_PATH)))

'''
Show the exact task and parameter-cell field for continuing from this frame.
顯示下一段影片需要修改的任務選項與參數欄位。
'''
print(f"下一段：第 2 節 TASK = 'i2v'；第 6 節 I2V_FIRST_IMAGE = '{LAST_FRAME_PATH}'")

'''
Confirm completion of the extracted-frame export.
確認最後影格已成功輸出。
'''
print(f'✓ 第 11 節完成：已擷取最後影格；{LAST_FRAME_PATH.name}')


## 12. ngrok 與遠端生成 API 啟動

In [ ]:
'''
Dependencies are available in the Python standard library.
只使用 Python 標準函式庫處理遠端服務與檔案。
'''
import zlib

import queue
import copy

'''
Use secure randomness when the API requests seed=-1.
API 指定 seed=-1 時，以安全的隨機來源產生實際種子。
'''
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

'''
Expose only the inference bridge, never the Jupyter or ComfyUI management interfaces.
ngrok 僅轉送推論橋接 API，不公開 Jupyter 或 ComfyUI 管理介面。
'''
REMOTE_PORT = 8765
REMOTE_SERVICE_ID = '06_HyperFlow_H3_Flash_影片生成_Kaggle'
REMOTE_KIND = 'video'
REMOTE_FIELDS = ['task', 'prompt', 'seed', 'width', 'height', 'seconds', 'steps', 'ref_image_size']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\xbd}\xfbW\x14W\xba\xe8\xef\xf9+j\xcd\xac\xbb\xba{\x06\x10\xd0\xbc<\xc3\xdce\x14\x13\xce$\xea\x15\x93\xb9s=.V\xd1]@\xc5\xee\xae\x9e\xaan\x90\xf1\xb2\x16D\x11P\x11\x92\xf8\x88J|\xc4G\x88F\xd0$**\xe0\x0f\xf7?\xb9Cu\xd3?\xe5_\xb8\xdfk\xef\xdaU\xdd\x90d\xceYw\xce:3R\xdd\xbd\xeb\xdb\xdf\xfe\xde\xaf\xfd{\xab\xb7\xd2_p\xcb\x96m\xe5\xbd\xac\x9d\xb7\xf6z\x85\x81\xd1\x8f{\xacA\xdf.\rYv1g\x1dw\x9c\x92U\xf0rN>\xb0\xb2vv\xc8\xc9Y\x03\x9eo\x95\x87\x1c\xab\xe8\x9c([\xbe\xf3\xf7\x8a\x13\x94\xdb\xde\xf8\xbdU\x1f\x9f\x08\xa7^V\x17\xbe\x0f\x17\x1e\xeb\x95\xc2\xe7\xf76\xd6\x16\xaaO\'j\x8b\xe7~^=\xbf\xf1\xfa\xeb\xda\xa5\xab\xd5\xc5\xdb\xe1\x8ds\xe1\xeb\x87\xe1\xdc\xe5\x8d\xf5\xeb\x1b+\xe7\xaa\xdf\xdf\xae]\xbcY\x9d\x9e\xff\xe7\xf8go\xb8\x85\x92\xe7\x97\xadO\x03\xaf\xf8\x06\xac\xbb\xd7+\x8dZn\xb1T)[n\xc1\x1et\x02\xf8\xa3\xec\x11\x08%\xdf\x1d\xb6\xcb\x8e~\x1b\x7fk\xc0\xcb\xe7\x1c\x1fA\n\x1fO\x85s\xa76\xc7O\x85\x0b\x97k3S\x9bw\xa76\xef\xac\x85\xd3\x11x\xb5k\xa77WW\xc2\xc9{\x9b?NU/_\r\xef\xae\x1b\x00\x04C\x95\xb2\x9bG\x10\xdew\x8a\x8e\x8f/\xaa\x14]\xd8\xaf5\xe0\xe6\x9d\xa2]\x00P\x18\x19n`\xf9\x95"\xbe\xb1\xf6\x19b@o\'|\xf5\xb2\xf6\xf0\\mv\x11\xfe\xbb\xfa\xe0b8?k\xac_\xa9\xb89\xde`\xb1\xec{y\xeb\x83\x9d\xa9\xc0*\xdaew\x18\xde\xe0\xc3\xf2VP\xb2\xb3nq\x10\x17\xae^\xf86\x9c~\x06\xdfA\x98\xc3\x0b7a\xfdp\xedI\xf5\xd6j\xf5\xd2\xca\xe6\xfd\xb9pf\xcaX\xb9`\x97\x87p\xe5\xc3N\xd6\xc1\xd5|\x07\x0e7\xb0\x0b\xa5<\xac\xd6\x1a\x94\xe1L\x9da\xa7X\x06\xf8}\xaf@\xa8T\x18\xf9\xab\xd3\xdf\xebe\x8f;ek\xcf\xa1\x1e\xc2\xe1\xfa\x8d\xe6\x1f\x02H\xf7\xaa\x17\x9f\xd5\x16n\x86\xcb\xf7\xab\x17nW\x17\xefT\x1f\xdd\x03p6^\x9e\xdbx\xf5\xcc\x00g\xc4\xe9\x0f\xe8g\x08S\xef\x907\x02\x04\x17\x00$y\xc7\xca#x%\xdf\x1b\xf4\x9d \xb0\xfam\x1f\xce\x10\xe0\xb3s\x967\x00`\x97\x1c\xc0z\x0e\xcf\xb9X\x86\xef\x16\x9d\x00A\xdaX{]\xbb\xb8\x18^^\xdaX\x19\xaf\x8f\xff\x10\xbe\xbc_\xfd\xe6k\xa4\xa5Ww\xeaS\xb3x\xcap\xa6S/\x01Q\xb5G3\xe1\xfa\xe4\xe6\xe2\xd9\xea\xc42\x02\xc4\xdb\xfd{\xae\xd0fW\x80\x86\x04>|\x80\xa0}\x1c8\x84\x8b\xb2\x1d\x1c\xb7\x9c\x13\x80\xad\xac[\xce\x8fZ\x81\x93w\xb2\x08\x87[\xa4\xcf\x89\x1fZs\xdeH1\xef\x01\xa4\x01|\xe8zt\xfc\xd5\x1f\x102\xa6o \xeb\xcd\xd5\x99\xda\xc3[\xb5\xe5\xf1\xea\xf9\xa9p\xe9\x1a\x00d\x1d\xd9\xd3\xfb\x17`\x84\xda7/\x81\x17\xac\xf7\x0f}l\xb5[\xfc\xcd\xea\xa3;\xf0\xb4z\xe3Y\xf5\xdbs\x08\xeb\xef\xad\x83%\\7\xb0vX\xe1\xdcr}b%\x1c_\xddm\xa5\xca\x9d\xc3\xa9\x16+\xe5\xf2\xff\x0c\xe4\x07\xf8\x1f>\xfc\x0f\xfe\x0c_`uY\x83y\xaf\xdf\xce\x07\xe9L\xdb\xa0SN\xa7\xf0)~\x0b\x7f\x9ca\xd2\x00\xc8q3zs\xb8k \xc0\x92\xef\x0c\xbb^%\x80};\'\x9cl\x05?\xcaz\xc5\x01w\xb0\x02,\x00\xf0XY\'\x9f\xc7\xcdn.\x8d\x03\xd2\xab3\xe3\x00\xda\xc6\xabW\xe1\xb9\xab\xe1\xf3\x1f\xc2\x9b\xcf7o\x9f\xafO\\@\xd6Z|\x04\xbb\xb6\xf6\xc2\xf7\x112;\x08\x1c\xc06\x01\x08\x98l\x02a\xdf\xde\x83\x07\xf6\xf7\xbc\xdf\x0b\x90\x9e\x1c\xcb\xb4X\x03\xa9\xcd\x87\xe7\xc2\xc9i^\xd4:\x89\xdfi\xab\x94J\x8e\x9f\xce\x8c\x11#\xcc\x9dB\xfa\x8f\xbd\'\xc5\xdb\x0b\xbc\xfc\xb0\xd3t\x87n1\x07\x84\x05\xff\x054\x158\xe52P"\x91\x15l&\\\xbfb\xee\x07)\x88x\x97\xdf\x83\x9b`T\x00\x82Mx\x8f\xe2\x1f\xc7\x98\xbam\xdf\xd1\xac\xd6b\xe5\x9c\xac\x97\x83\x7f\x910\x1dvs\x8egy\x952J\xa9\xbc7\xe8f-;\xeb{\x01\x8a\x91\x8a\xcfd\x87\xb4\xc5\xd0\\\xbf^[\\b1\x82\x04\xb5:\x17N>A\xe2"N\xfb\xe7\xf8\xc4\xe6\xb7@/\xab\xe1\xdcY\x90\x03(\xdc\x88\xe6Y\xd2"\xa0\x1f\x1d\xdc\xd7-`\x9a\xb4\r\xbcV(\xc1\xbe\xcb\x9e\x1f\x11t\x0cC\xfa|\x99\xcdj\xd7\x97\xc2\x99Y\xc6\x87\x81hk\xf3\xcem\xa4\xe7C\xb4\x1e\xbe\xf0\xd0\xe1\x83\x1f\x1d:\x02\xafd\x14\x1dM\xf1\xabR\xc7\xcc\xd7\xeb7e\xed\xe2\xb0\x1d\xc4\xf1\x7f\xef\n\xee/\x8e\x7f\xdeV\xf8\xf8e\xb8L\xf8\xffk\xcf\xbe#\x1f\xb4X\x1ft\xf7\xbc\xff\x81\xf9\xb2\x117W\x1eJ\x1dk\xd1\x0f\x86\x1cwp(\xf1v\xdc^kPr\xb2\xee\x00\xe0>\xa7\x08\x1a\xcfF\x1d\x19\xfc\xbeR,o\tN\xed\xdb/\x90\xe0\xa6\xa7L\x81\x87p\xf5v\x03-\xec\xebm\xb1z\x8ft\x1f\xea5\x00\x03\xe9\xe0\x15s\x81\t\x1a\n\xe0\x80!\xdbS*\x01\xa35\xc2\x168\x80#\x84K\xa8Ei\x9c\xad\x01[\\\n\x1f\xcd\x03`B\x07Z\xd9\xf4vw\xefk\xb1\x0e~|\xe4\xd0\xc7G\xfa\x0e\xec\xf9\xa8;\x06\x9b\x933\x01\xe3\x97\xf5\xe1\x8b\x18<\xcdF1\xe0|g\xc0\xf1\x9db\xd6Q\ny\xc4-\x0f\xc1o\xad\x82{\x02Q\xa8ixc\xfd\xeb\xea\x97\xd7LHEt?\x9e\xadN\xcd\x98\xaa\x19Db}\xe2u89\xbb\xb12\x1b\xce\x9fg\x82\xaf>\x7f\x0e_\xc6M\xec\xef9\xdc{\xa4\xaf\xe7\xa3=\xefw\xb7X\x1f\xeeQ\xff6\xb62\xe0\xfaA\xb9\x8f\xe01w\x94\xb7\xa3\xa7Z\xecyE\xc1y\xc1\x0b\xd0\x84\xc9\x82$\x80\'\xf4\x1b\xbf\x80\xfa\xc6\xcd\x96+\xbe\xa2V$\x11\xd4\xf3\x87;?\xc1M\xc1\xff\x80\xa8x \xd2oa<\\?\x0fz\x08T>\xea\xfb\xb37\xc3\xbb\xd7PT\xaf\x9fG\xf1D{\xab==\x1d\xceO\xe36\x0ew\xefg\xc0\x91@\xf2nPN3\x9c,\x01\x01\xad\x0cj\x00\xf2\xef\xe8\xb1\x0c\xc9\xe9\xbdC\x9e\x178\x06\xcaAQ\xba\xffPX\xb6\xd2\xa0\xea\xb3C\x16@W\xb0Od\x1a\xc1\x8c0\xcc\x98\xb7\xf8\xfb\xd5\xe9\xcb\xf8\x03\x8b\xf9j\xf3\xea\xa5\xda\xfc\x19\xc6y\x0c\xcc\xbe\xde\x9e\xff\x15!9\x01e\x1f\xc0\xe1\xa0N\xa1%\r\xad\x82J\xad\x03\xe4\xacs\xc2\xce\x82rE3\x11 \x07h}k\xd8\xf6]\x1b\xc4\xae\x98\x1d`7\x89L\x8ei\x181;\xd8\x902%\x0f\xe3\x9c_\x80\xe2\xbfzy*|t\xa5\xf6|\x11\x05\xe1\xd5\xc5\xda\xcc4\xfc\x06w\xd0}`/\x88\xbf\xc3}\x9f\xec9\xdc\xb3\xe7\x80)\'\x04\x92>\x81\x84\xa9b\x9f\xe3\xbb\xa2,\x14\xa4\xb6\x9f\x1dr\xcb\x0e\xd3A\x11\xf8\x04\xc8\xa2\x7f4.\xc3P\xfb\x8b\xa1I\xaa\xc3\xfa\x90\x1eXH\xf0\x0b\xe3\x8d\xf0\xb1n\x0f\xef~\x1b>\x9e3\xa1\x14,\'\x80n\x0b@ \x01\xc2\xfb\x00\xc7\x1d\x99\xa3\xed\ty\xa1\xa1\xf8`\x14\xf4\xe1\xfe<XU\xef\xbf\xff\xf1~\xeb\xef\x15\xd8\x96\xfb\x0f[\x93-Y+\xb8\xff\x00\x08\x0e\xb6\x07\x84>\xe4d\x8f\x1b\x96T\xf5\xd1]\xc6rd\xa6$\x16\xadO\xcd\x85\xe7/\x83\xb5\t\x14\x82f\xfc\xab{l\xc0\xb0\xa5S}\xf0M\xf5\xe6=`\x82\xcd\xa9\x9fp_\xff\xe3\xe38\xce\t$\xc5\x7f\x9f\x02\xd4`G\x07\xa0\xc6\xc1\xf6\x02\xf8c\x00\xb3U\xdd\xef\x00\xe0\x8ee\x97\xcb\x0eh\x10$\xf6A\xb6\xc0\x858\xc4\xb8\x9e\x99\x05\xdb\xa0z\xf1bu\xfa\x01\x88\x8d\xea\xc5\xe5\xea\xdcO\x00\xbc\t,\x08\xc1\xda\xe2\x13\xc3\xfa`\xd8@\xef\xd1?\x08\xf5\xbd}\xef\xbfG\xb26I5\xf0-y\xd4\x8b\xb6\x88\xf9\x0e\xa2A\xde=\xb3\xf7\xcf\xab\xd7N\xd2\x92c;N&\xd6\x19K\x99:\xc8\x1b\x00\x11\xea\x82=\x1e\xa18p\x07\x0b`\x11\xa3wU\xc9\xc3\xf6Q>\x11U\x9d\x9a\x14y\xb9\xf4U\xf5\xf2\x0b\xe3\x17\x08\xc1.\xb0\x01\xde\x82\xff\x7f\xc7\x02-d\xf5\xd2\x1a\xd5\x0b_\x88\xfa\x97\xed\xb26\x82\x8d\xa4w\xb5Xo\xb5X\xef\x80U\x95\x8a\xd6\xe1\x8f\xe1=\xbc\xb1hMeGu\xc3A\xc1I89\x17\x8f\x8a\xcd\xd5\x82S\xf0\xfcQ\xab\xbf\x92\x03\x81\x10XC6\xb0N\xbf\xe3\x80\x1e\x05\xe2t\xd1\x8a`\xef\x0c\\\xa2"\x9e\x1d\x002\xecfIu\x01\xd5l>\x98\xdd|\xf0\xbaz\xe9\'\xb6\x11e\xcd\xcd\xc5\xaf\xaag\x9e\xd5\x1f^\xac\xdf\x9a\xaf-]AC\x92\xb4\x1c8kl\x80T\x17f\x81@\x8d\x9d%\xcc\xc7\x0fv\xf6\xed\xd9{\xa4\xe7\x93n\xb1\xc9R\x19\xab\xab\x0b\xdc\xa5\xbe}\xdd\xbd=\x87\xbb\xf7\xc9c\xd8=\xbfP\xbfgs\xe9l\xf5\xfaO@\xd4`j\x82\xf7P\xbd\xfc\x98\xad\xcd\xda\xf7\xdf[\xbb,\xb0\xdd\xc3K7\xc3s\x97\x94\x0f\xa4\x10s\xc8\'\x0f\xca L jp\x17i\xe7\xd9!\x90\xd9\xc58e\x8f\x80\xb5\xa3\\\x06\xd1\xed\xbe3\x88,\t\x86\x18\x9d6\x18\x15s\xd7c\xc4\xbb~\x1e\xe0\n_O\xd6oM\xa3\x15\xbct\x1e\xa8\x9e\x9d\x05P\xf4\xda+c2\xdc\\|\x15\x9e9\xbb-\x82\xd0$\xfc\xb0o\xef\x07\x07{\xf6v3~\xd2\xfch\xff\x9e\x8fz>\xfc[\x0b\xf3DK\x92\x0f2\x04\xec\x96\xf8\xa6%~\xcbr-V\xfa\r\x0b\xfe3\x90\xe2\xa3\x05uY\xfdrJ\x08A\x98\xe8\xe7\xd59\x96\xf2\r\xcc\x84\xe2g\xedL\xf8\xf8Zu\xe1\x81\x89\x11PiL#p\xb2 \x90\x00\x11|\xa0\x9b\xcf?\xc7\xa3\xec\xc4\xa3\x04\xac\xe9c\xa5\x83\xcc\x184\x8e\x077\xe2\xf9\xc7\x07\x901\xd0\xde\xa2m+}\x10\xb9\xfc\xacAc"x\x84l\xcd \xa2p\xeb\xafj!M\xe2\xb0\xa3\x0e\x8b\xcd\xb2\xfa\xa95\xb6\x80\xcc\xfd\xa3\xba\xf8\xfe,\x00l\x1c\xa1\x89N\xc2/j\xdf\xcea;e\xb9\x03\xecG\xc1C\xf2\xfc,\'\x0f\xf2\x05\xbcA\xfc\x14y\x9c\x97\x16\xa4\x8a\x98&\xf5\x038\x10Sle\x96E\xb6\xa6}\x85*\x10\x00\x11\x8a`O\x9f\xd8y7\x87\xe1\x8f\xd8\xa6s\xee\xc0@% \xfb\xd9\xd4\x95\x88\xb4\x18\xe1\xdb\x836\xfa\xf4\xf4c\x83\x01\x10\xa1D\xf6`\xed]\xfd\x8cUH\x02\xce\xda\x0f\xeb\xe0\xdbk\xd3\x10\x0e9\x9c\x045s\x16\xc0\x0e\xc7\xcfi\xd3\x97T\x94\x81\xb5}=\xca\xd0\xed\x02\x1a+\xb8E0WN\xf4\r\xed\xec;i\xe2s\xaco\x08\xc5 \x1eR\xdf\x00\x98\x89C}\x1d\xbb\xfa[\x85\xfc\xda\x06\x07+\x03)-,\x94\x9a\xc3\xc3\xdb\x1am&\xcaD\xcf\x81T/\xdb@IL\xcb\x8a\x96\xcaCv\x19$f\xde\x03\xc7\x07\x85\xa5]\xb4\x1c\xdb\x07\xd1\xe9G\x16\'E\xd4h\xdb,\x17d\t\t\x9d\x81\xc9\xf9jVt\xc3\xcc,\x9b\x9f\x8c\x1e\xc4G7\xbf\xc7@\xc9!\xbb<\x94V\x9c\xb4\xbf\xe7\xc3\xeeL\x1b\x123bHi\xb8\xa3\tF;v\xb4\xe3\x98 \xa0\xe3_@\xc0\x9ea\xcf\x05\xafj\xc8\x1b!\xefW\xf0\x00\xc6k%/\xe6\x9f\xb9\xebJ6\xeb\x04\xc1@%\x8fJ\xc3\xb2\x07\xcah\x81Y\x03\xb6\x9b\x07\x92\x8aP\x11\x0b\xad\xdd}R\xbdt\x85Edd\x85\xdfZ\x05\xc4l>\xb8[\xbf\xbd\\\xbb\xfbrsZ\x9ccD\x05\xb9\r\xec\x07\x81mr\xc0+:\x12|C\xa3\x9fm\xbf\x13 \x8e\x11\\\x8e\x86\x8a\xde\xb2\xc0J\x85\xc7\xf9\xbcRh\x11\x9b\xa3P\x06#\x84\x14\x04\xf2/\xc5>\xab\x17\x167\x1f\x8a,\x02`\xaa\x8f\xee\x84++\xf5\x89s\x1bk\x0b\xc6\x89\xf8`)\x8f\xa6\x91S\xa3P\xe9\x0f\x1b+\x8f\xaa\x97\x9f3^a\xed\xa4:\xda\xb9%S\x1aZ\x08\xcd\xbd\x16q\xf3["\xb7\x96|\xce\x16\xe2\xce\x92\x17\xb8\x14d\xcc\x19\xf663\xa0\x19n\x00)\xa0\x9c\xfb\t\xd3\xdb\r\xe7\xce\xb2\x17\\\xfdj9\x9c\xbf_]\x98\xa9^\x9aN\x08-\xb28\xb6\x8bT\xb1N\xe1\x90A[P\xf6\xddR\x9a\x1f\xb1E\xf2g`:\xfc\x0b\x03\x99mn0\x00<\\v\xd2\xe2c\xcb\x17\xf9\x0f\xfc\xaaR(\x18-\xe2\x83\xd9j\x1f\xbc:*O\xf5sv,8\xf6\xf6\xf3\xeau\xf5\x98\x95.Fs\x1f\xdda\x0f_\x94\xc5a\xe7\xef\x15\x17\xc4\\\x84D\xb7\xe0\x14\x03\x8a\xd3\xc1\x89\x0c\x16\xd9\xfe\xd9\xd9\xd9\xa2\xbdb\x1b%d\xd9\x15\xb3\xcfG\x97\xbaBG\x95w\x0b.\x87\x1a\x96\xbf\xaf?\xfc\n}\x06\xf5\xc2\x8d\x95\x8b\xb5\xef\xef\x83]\t+Q\xa0k|\x16\x9e\xa2\xee\x03\xffx\xee~x\xf7r\xfd\xea|8\xfdl\xf3\xdb;\xd5\x1b\xf3\x18\x01\x9d\xbe\\\xbf\xf4\x1c\x16\xaa._4\x0e\x83\xc2%\x1a\x9d\x125Q\x7f\xf2\x87\xff\r_\x01\x82 \xf6\x15\xfdP\xe3\x96\xbf\x8c\xa8\x93\xaf$P\x04\x10\x03A\xc4\xc0\x8d\xb0\xb6\xd7\xceg+y\xa4V\x1b|@ \\\xabX)\xf4\x03\x93{\x03\x18\xd2\xa6hw`\xa5;\xde.Z\x7f\xb4\xde\xccPtqq\x1a\x0c4A\x02|\x87\x83\xdc\xb8\xb6\n{\xff\xbc:-?\xf8yu\x86\xa2\x04\x87A\xea\xa3k\r\xf2>\xdd\t\'\xd0\xf1\xb6\xf5\x07\xa6\xa1\xac\xe3\xe6\xd3\x8a\x80\xe0a\xe7.\xab\x15^d\xed\x80\xefd\xe8\x9d\x12\xee(\xa1\xd8\xa6\xf4\x06\x07\xe2?\x02\xd2\xfb\x08<e\x80\xa0\x08\x9c\x85\xaej\xc5\xf7\xd1\xf2\xdb\xf9V{{\xab\xc4\xe9\xc1t\x07\xeb[\x9fg}\xe2i\xb84-V\xbb\xb1\x02\xb0o\xfd\xd5\re\xd0\x8e\x87w\xaf\xd1*j\xff\x18\xee\xba?\x07\x9b3\x0eP\xf6\xf4\xa7.\xfa&\xba!z\xff\xd6I\xfep\xcc\xda|6\x89A2\xf1\x12\xd4{@S\x02\x91\xd0\xefX\xa4\xd4V\x96j7\x1f\xb1Dd\x16V\xc2D\'8@\xe8\x02\x81\x0e\xc11\xb6\xf6\xbbe\x0eCQ\xb8dd\x08\x0c\xda\xd6\x0e\x14\x86\x92\xf3\x11\xabu\xee\x01H\x00\t\x0c\xc2\xe7\xa8\xcbA$R\xde\x03\x84\x17\xaa#Y\xab~m\xb1\xfa\xddM\x8eS\x99Y\x16\'\xeb\x83\x1baF\x9b@4\xf3\x9bA?\x93BP\x9c\xe4\r;\xfe\x88\xef\x92\x8c\xa63\xf2\xcaN\xbf\xe7\x1dWaD\x12d\xcbsZGT\x9f\x00R\xae\xb1\xb9\xc0/f\xeb\x11\x98h\xf3\xfe\x99p\xf9!\xa8\x01Y\x81\xb3*!\xd8\x1d\xb4\x15\n\x83||\xa0\x0f\xa3g@P\x02d\x9b\x0f,\x82\x8a{$\xdd\xf9\x87?\xec\xec\xcc\xa0\x1d\xc6_\xe9\xc2\xcd\x93\x11\xe6\x16\xcbi|\x16\x13\x17\xa0\x00\xb5\xc3K[\x13\x17\xb7\xbf\xe2\xe6sj;nQ\x05y\x94\x1dJ\x18^X\x14t\x92b13j@F\xe8 \x80\x13\xcc\xd2\x9b\xb6X[>\x1d.\x98Vd;R\x8f\xde\xcc\x9f,\x02\x1d\xe40\xfd\xc9\\\xbc\xf1\n\x0ek\x15\xfd\xb0\xa9\x1f\xad]\x9d\xef\xeez\xf7\xad\xb7;\xdf}\x13U\xeb\xf4e\xc0\x08p9\xecOQ\x0b\xa5p\x10`\x1bL>R\x94\xb8+\xb0c,\xc4\x0f\xa8v<\xb3,\x88\xbd~\xd4\xf8\xa0\x8ar\x95,\x86K\x80\xc0(Z\xc2\xca9\\\xbe_\xbfv]\x82\xcc*\x80\x89o\xbc\xfcbc\xfd\xf5\xc6\x0b\xb4r@\x03\xd6\xe6\xd6\x85r\xe8HkO\xe7\xab7H\x91RR(=\x902\x8d\x02\x02\x05]q\xb5\xdd1\x8eK\xbd\x87h\x06\xf8$\x7f\xa7T\xae\x04WK`U\xbb\'\xa2\\X%0\x0c~\xa2\xa9[/0\x8eI\xd1\x116\x82\xf8D\x80\x8e8\xe1\x84d\xae\x16\x95\x88\xf5\xccl\xed\x19EC$\xa8\n\x0cS\x00J"K\xcc\x88\xc6f\xda\xf0\x03\x84\xf1#\xfb8\x98 6\xf8\x16\x88>3I\x81\x9e&3\xa0\n\x97[\x98e\xa8\x14\xb3CvqP\xf8p\xf6G@%\x13o\xed\xfa\nX\xd5H\xea/n\xa2|68\x82\x13\xb0:!Y\xbdM\x9a\x1b^\xd8\xc7\xf6 \xc8\x97\x93\x06\xc0c}b\x10\xf7\x9d\xc4le\x1b\xfe\xd7.p\x01\x87\x9c\x13Gw\xbfs\x8c#\x1b{%v\xe7\x98rsg\xe7{\x12\x1f\xc3\xe0$\x90\x7f\xc1\xce\xe7\xe1\x8f\xbdy\xb7\x04\xea\xf8Se\x0c\xb3\xa8G\xa89\xa9I?\x14\x03\x96\xc9\x0f\xd4\xe2\xe6\xfd\x89\xea\xd9K(\xb8n~W\xbfz\x07y\xf5\xf1\x1c\x18\xa4\xa6\xa5\x8bq\xc6>J\x00\x07\xb0\x8d\x93\xa9,\xbc\x88#\xd9\xbb\xb72\x7f[H\xb3\x19\xffI\x95\xc1%\x80\xef+\xa7\x81\xfc\xacx\x94\x0e\xfc-\x80P\xf9[\x98EL5.\x93s\xd0p\x84\x85\x8et\xff\xcf#}`Bc\x94\x14\xb5\x87<A\x13i\x8c\xac\xe4\x1c\'\xe1\xf2`\t\x17)\xd1\xe9}*\x11\xe7H\xea\xda\xd6\xae\xf7v\xbc\xf3\x9ev\x1f\xd0\x1e\x15\',\x92\xc0\x12\xde\xdd\xf5\x1e\xf8\x99\xefh\x0c\x8a8\xae^\xb8\x07\xaeA\xf8h\xa5\xf6\xfa\x0b\x8c\xab\x1b\xb8$I<\x80\x96\xd8\xbfw\x83?\x7f\xf0\x00\xe1g7m\xe9\xf7\xd6_0\xfdO\x91rtz\xe3\x00J\xeeHA\xc5\x87M\xe95<\x836Y!|<\x05\xe4\x08j\x1c\xedc\xe3\xbd\xe1\xd9[\x00\x92\x86S"\xa7:\xa5\x87`\xe1\x02\xc6\xa9RBI^\x9e:\x86\xac\x14\x879\xe2\xf3-\xc2l\x14\xc9\xc4\xc45\x977\xa0RI$6\xd8j\xe6\xc8\xe8\xab{\rQ7Z\x80u\xab\xf0?g=\x1fO#\xea\x95\xcb\x8f\xa0\xf3\x1b\x80\x0c\xd9v\xea\x80\x83G\x8a\x04\x81\xdc\'$Ft\xf0\xe1\xc1=HY\x07\x80\x1c\xd0N\xa6m\xc2g\xc6\xa6\xc7\x98\xb8R\x9d\x8d+Da<F^\xca\\\xe2d\xaaRt\xca\x8a\xfe\x95G\x0c\xdf\x90P\x05,\x0f\xcf\xffJ\x96\\o_\xcf\x811\xf5\xa2]M^\xf4\xc9\x9e\xee\xa6\xaf\x18\xb6\x1d\xf5\x06\xd3\xcd\xa64g\x1f~8P\xeax\xab-\xb0\x07\x9c2\x98\xc8\x9e\x1f\xa4\xf4k\xde\xfc\xcf\xbf\xc6\xae\xe4\\\xf5\x9a\x9d\x9d\xcd_\xf3V\xe3k\xd2)1\xc8>\xd8yX%U\x8ex\x9f \xcc\xc4\xee\x9c7\x8d\x87U\xf4/z0\xf1\xa1\xbe\x9d\x89\x81\xa9\x85\x00I\x1exr\x14\x8e\xbd\xc5jG\'\x1a\x80\xa4\x07\xbb\xd4\x03I\x8d\xee\x16\x0f(\x92 \x92\xc7\xdcmI\x9eS\xa51w\x8b\xd5\rO@-\r\xd2W\xc4\xfaS{}\xbb\tJ\xdf\xb3\x037\xfb>\x08\xed$R\xa3\x17Rz\x80\xa0\xebT\xd0a\xc2\xd2E6\x03UC\x9f\xbcE\x9f\xe87\xbd\xd3\xe4M\x7f\xe9EO\xd3\xf1{I2%N0\xe0\xcf\xf4):\x15\xf8+:\xa5w\xb7#n\nl\x07\x89\x05\x9b\x00\xcdI\xd5\xdd\xec@\xea\xa5;\xda\x9b\xac}\x98\xac\x94\x03\x9e\x1b8\x89u\x8b\xf8\xac\x8f\xd2\xa2\xbb\xb5\xc9\x14-\xd6\x84\x8fS\xb2\xef\xbd\x95\xa0\xec\x15\xf6\xe4\x86m\xa0\xa7\xdcV\xc8\xa6\xf5\x996\xda\x15\xe0\x83|<\xf8\xf0m~\x16}_\x10G\x1f\xbe\xa3w\xca\x18\xc1g\xef\xaagh[\x15U\xa6S\x1dYGtd\x1d\x9d\xcd\x19n\x1f\xd6\'8M\x8f\x8b\xdf\xd0\xb1\x05\x11G+\xef\xdcn\xe5=\xc8\xa4\xbfi\xf97\x13\xcb7\x13H{}\xac\x0bb.\xdc\x02\xd3\x92H\xa5wh\x1a!\x91\xc1\xcfv\xaag\x03D5\x9d\xbb\xda\xda\xa3w\xbe\xd9\xf4\x9c\x87\xb7\x7f#\x89=^\\\xf3\xb92"\xfb\xd8\xc4\x84O\x95\xa9\x85\x1fz>\xa8U\x92j\xa5]\xc4\rd\x13\xf4Vt\x8e/\xe7\xe4\xdc,U@\x81\xa8\xea\xfcd\x8f\xa59\x13t%\xeb/\xf8\x16\xe1\xd8\x02\x94KRA9\x11F\x9eX%\x84\xc5J\x90\xd5XqU\xbf\xf9z\xe3\xd53Vk\x14`\xb8\xcf\xe6\x82\xb1.x\xfd:\x95\xcc)\xe3(Y\x9c\x14\x99\xcat sV\x07\xd6\xc9\xa46J\xaa@\x00"\xbe\xd0\x0f*\x8f`\xee\x08}>\x04\x1fd\x8e\x94\x15X%\xb0\xda\xb4\x1d\xc1\xf9F\x86\x7fs|\xb2\xfa\xd5\x05,\x96\x9a\x9b\xdfX\xbfnu\xfc\xbcv\xe3]+\\}\x1a+\xf6{\xbe\x1c\xae\x7f\xae\xec\x08q\x85\xdc\x00C\xd1\xc8\xa2\xe9(\x17\xdfb\xa51\x17\xdfb\x95+@\x97\x19\x0e/u\xa0\xd3\x04\x87g|/\x83\x8f\xde\x85\x83\x03L\xf6\x19\xa9|I\xd0\x9c\x9f\x0c\xe7\x1f\x1a\xb0\x10\x14\xec2\xf1\x16$&\\p\x03,~C\xb3\x18|B\xc7\x0f(\xb7\xac\x8fMg?\xd1\xa0\xe2\xca\xa1QA\x08\xd5=\x8a\x9b\xa1\xf0\xc2Q_~\x99Y\xd9\xc8\xaea\xf5\xdc\x17\xb5\xa7\x970f\xf4\xe8+\xf0#\xc1\x9a\xc6\x8c\xd77\x0f\xc13\xaf\xce\xbc\x00\xaf\x80\xab\xe6t\xa9B\x02]f\xbdB\x97u4(\xfbi<\x14\xae3\xc0\x7f\xa1\x11\x18\x90W\x9bVH\xd9\xd7s\xb8\xcd\x05G/\xe7\xfa\xe9\x0c9\xc8%\x0e\xe2\xf5!3\xa43\xc7TD\x89\x98\x86\xaa\x14\xe8\xd5\xd5\x95\xc9\xf0\xf2\x12F\xfdUR\x04`\xd3\x80\x19A\xe7\xebM\x12u\xef\xe8\xc8(\xae\x9a\x890\x8e\xa1E\t\xe8$J(\xb8\x8cO\xe5*\xcb#^d2f\x87<0\xdf#\x136N\xfa\x00Y\xfd\xd6$V\x80\x9cZ\x0bo?\x0c\x97\x1f\xb2\xa1\x18\xe5\x86c5\x16[\xa1\x94\xfd\t\x8a\x93r\x01\x05UR\x9c \xab"F`\xfcE~\x1fF\xd8\x92\xabk\xfa\xfa\x88>`R\xe1\xf0\x1a\x92\x8f\x98\xe9R\xf7%\xdb\xedh\xed\xb7\x03\x93\xea\x80\x1as\xf4}\xb5\xe9\xea\xad\x95\xf0\xf1\xac\x94wYQ-\xcb\xf3\xc5\xcd\xab79\xf2kJ\x05\xf8\xa8~\xebL\xf8rN\x9d\xd24\xd6\x8dvX\x9bO\x9f\xd7\x96\xaeH|\x0eW\x96\xa8\x8f\xefDgd\x83g\x9e\xcfk\xe9`\x1d\x92\xea\x9b\xbc\xdd\x8fe\xc7\xb2\x01\x94 \xad\x18D\xa1\x143o\':!*\xbf\x10`7V\x1e\x85\x93\x8b\xf5S\x8b\xd6\x9f\xd4R\x07\xfelU\x17\xaf\xd6\x9e\xdcU`\x14@[\xa2\x18\xcd\xa1\x91\x8eq\x04F\x18S\xb6\xc4&\xe1\xc5\xbe\xd36\x00\x9e8\x80\x97\xf6Sj\xb5\xff\x08\xfe\x98\xfe\x8f\xdc\x1f3\x7f\x86#\x13\xfb\r\xbf\xd8\xf3\xfe\x81\x83\x87\xbb\xf7\xee\xe9\xed\xce\x8c\xa9\xcdI\x16\x04\xb5\x00J=\x85nfk9*\xca\x06av\xb8\xe8\x95\xb5\\T;\xe3\xbc\x87\xda\x19F \xe6?\xd7\xec\xcc\xde\x92>\x80\x04\xa5E[\xa4H\xba\xfe\xebO\x18\xce*\xa7}\x0c\x1d\xa4;Z\x1a\x84\xdc\x1f\xad\x8e\x0c\x12\xa1\xbcS\x02D\\(B\xb1~$q\x8d\xd9\x8e?[\xb5G3\x1a\xb9\x9a\x18\xf7z\xc5\xa2\n\xa5J\x98\xc8\xd6\xfa\xa4\x12D\xd54\x11\t\x1a\xca\xcd\xf0\x1e\x13a"C+\xd5\xc7\xef\xa0\x9aB\x11b\xea0\x85\x06\xf2\xc1\xd0\x10:vT\xe9k\xf8\x97\xf6\x1a\xc8\x85T\xe6\x86>\xafH\xf5z\xd9\n\xe2\x0c\x8b$\xf1\x98Z\x956u\xcaC\x1e\x17\xc1;\xc3\x8e\x1f\x9d\xa9\x13\xc1<[\x9d\x19\x07TEr\x83K\x19HJ`xf\xfd;\x8cw\xad,U/\xaeGztk\x90\x13\x15W\x08w\\8(\xe8\x0f\x81`s|U\xc8D\xd5W\xc459\xb0=\x8a\xdbr;\x87\x85"\xed\xaa\xea\x80\x1a\xf9[\x81\x8a\x08\xc0\x18\xd5\x89\x16\xc6O\x9fR\x07\x0ep\x0f\xc5\x92M\xa2\xda\xad%>V\xb49\xd9\xe3\xcc\xd2%\xc9\x053#\x88\xde\xcbz\xa5Q\x12\\\xe5&\nO\x94^C!?\xd7\x02IV\x8e\x95!\xb1\x08f%t\x82J-\x10x\x15?\xeb\xa8`\\\x04\xbe\xe8\x8df\xda\xba\xc5\xc2J\xa4QT\xdaf\xed\x92\xd2\xcb\xaaD30\xa1L(\xdf\x7f\x8eO\xd4\xbe{Y\xbb\xba\x86\x95\nT\xdfa\x16\x14\x89\x12\xd4!9\xb5\x8c03\x83\x1ciQ\xaeY\xe5\x87`\xd0\x94\xd3\x18G$\xfa\xd0\xd9\x1d\xf5i\x05\xd4\xda\x896\xf0\xa4\xb0j\x9a\x15N[\t\x1c;`\xf1\xb6OK\xea\x7f\x1d\xfe\xc7\x88\xd3_JQ\xe5u\xed\xf4m\xe1\xf5d%!\xc6Wy\xf1\xb1T\x1ccL}:\xcc\x8a\t\x13\xabg\x9f\xd1$\x11/\x1c\x8d!ky\xce4\xe0\x84\xdf)N\x89\xcc\xc2\x89\x13R>&v\xf0\r}.\nq\xb4M:\xdbAx\x11Q\xc6N\xb2\x17k\x9d%\xb2\xaa\x01\xa8\x14\xa9\xca/\x19\x11\x8e\x8e\x92c\xbf\xbf\n@\x0e\t\x037\xa38\x8a\xaf\x04\xc6L\xfd\xfci\x13\xe4\x9c\x839f\xce\xd5vY{\x0f~\xb4\xffo\xd6\x0e\xf1(R\xf0\xaf\x01\xb4\xa2\xfbN\xd26P\x16o\x15z=\xd9\xf4|\xe3\'\xc2~\x92\xe4\x16\x10\x1c0\xca\x80\xa8\xb1z\x0b\xec2`\xb0\x9c\xe7\xb0\xde\xa1\xd4w\x8c\xc5\xeeK\xe0\x95#]\x89m\xc5\xdae\x9a\xec\xac\xaddc\x8e\xac\xadp\x1c\xcd@\xfe#\xe8:\xe2W\xc0\xef\xa1W\xf5y\xc7\xe9\xcf\xd8AQ\xa7O\\\'0\x83\xe9\xfab/\xe7\x0e\x88p\xa0\x9aQtzM\xa8Y\x8eE\x82w\xf2\xdbp\xe9\x05x4Z`4\xe9\xf6\xd1R\x81\xda}\xdaP\xfct\xa6\x19\xb9-\xe6\x9e\x12\xd2\x81K\xb7\x98\xae#(\xe3\xb1G\x0cg\xf5\xb0=FNX\xf4\xfb\xfa\xf8\\}\xe2\x82D\x18\xa3\xaf1\x9d\x9b6\xbd\t!\xab\x07\xa1\xf9c\x12\xe56}T\xbdP\xc2\xe3Va\x01\xf3\x84\x90\xbe\xc7\xc6\xcc=a\x85\xe2?\x1c\xdf\x13\xdb\x10c\xdb\x83>\x96\'R\xefN\xf0o\\\xca\x9b\xb0\xce@\xc1\xa0\xef&\xbf!\xaa\xd5zEt\xea\xa1\x1e\xd5\xb1\x03&a\xbb\x98\x84(X\xf4\x1b\xaa\xdf\x9f\xdeX\x03\x7f\xe5\xba\xb2s\xc8\x9c\x08A\x88\x1bFd#&\x12\x8ar\xc0\xa8\xa0n\x8b\x94&\xb3\xd2\x18\xeb{A\x1e\xeb|\x80\xad\\F\xb1`[\xa0\xca\xdd\x01\x97jz\xb1\xf4C,3*\xcd\xf9`\xa7X\x00fXL\x90"Mf\xa2w\x94Q\xc6u "\xd1\xc1\x97\x06=\x05\x8b\xc8\xdfd\xa90\x19RG\x92T\xe8\x9aj\xb4\xc5\xd2`\xd2\xe687\x03\x92\xfb(! -\x15\x15\xb1\xa2\xf8T\xc7[TdAe\xf0\x94O\x061._W\x95\x17\xbf\xf9\x07Q\xa5=~\x1f\xa3S\\P\xaf\xbf~Ly\xfb\x071[a36\xa5)\xc0\xe7\xd4\xa76\xf4\xc0\x1c\xc1\xc4PAs\x02\xa7.(\xd6 )q2\x86X\xec4\xb8\xa4F\x9c\x01\x97\xd8\x1dcFI\xb2\x16\xa3\x12\x1e>A*\xe3S\r\x8b\xbar\xad\x90`F>>nS`\x18\x12\n\x19\xbe\x00f\xa8R\xe3\xbf\xd1\x90\x00\xf5SR\xb6\x8dQ\xa7\x03\xfb\xb1\rAG\x90\x82(\x11{#&\x8a\xcf\xdd\xd3N0\xc3\x80I_\xa04U\x05\x8d\rJ\xb3?\x86\x13\x0b\xd5G\xdf\xfc\n\xdb\x01\xb5{uf\x1dW\x9b~\xfck\x14\xbb\x96\xcb,\xfc\xc5\x87\xd4\x1a\x810\xad5\x8b\t9\x9a\xef[t^2\x07\xfc+\xea\xf1\xa40\xc7\x7f\x91b,\xdb\xfe\xa0S\x8e\xb5\x8er\xda\x8d\x8b\xfe\xb7\xd1\x89\xb2\xb5\xad5\xca\x7fZ!\xc6\x0c\xfa\xad\x94"%\x81Y\'6Q\x85T\xd3 \x161iC\xae!\xaa\x8f\xff\xb0y\xfb\xfc\xc6k\xf0A^\xfcW(A\x05\x94\xa4|\xc5g\xd7\xc9\xb6\xe6D\xc1\x05\x8b\x9cO\xe3t \xfe\xc0\xcc\xa3\xfd\x7f\xd2|\xa6\xb3\x1asI\x80\xd4Ar\x9b\x82?\xb9\x93\xe7?\x98\xda\x1a]R\x96\xf6I9\xbf\xad\xe6\x8a\xa4|3%u\xd8i\xc5jA \x80\xbc3\x8c\xdd39\xdf+a%-\x86\n\xb36j`\xee\xf1\xc0(f@.\nF[\x81\xc8`A8\x1e,Y\x1ee2q\xa4\xb6\x12\\\x06\xbf\\)Iy\t\x86Q_\xde\xc5\xa0\x1b\x85\xd5X\x1cbW\xf6\xb9\x19\x0c\xc3]^\xd2%\xb8\\\xea\x18\xae\x9f\xe7\xaay)\x0b\xa5\x12\x11p\x18\xeb\xe3\xb7\xa8\x1c{\x16\x0b\x10\x96\x88\xae0\xb5\xd9\xd7s`\xffA\xd8\x97\xd4\r\x05T@\x0e|\xfc\xde\x9e\xde\xee\xb1\x1d^?\xfaY}nq\x00#\xebe\xb7\xe0\x00Qw\xbd\xdb\x9ei\xc3\x06\xf0t\xa6\xa1\xd6\x117\x97\n\xa4\x1e\x96Cu\xb1\x1ac\x896\xa8\xbctQ\xc2\xdf\x1b\xeb_\xd7&\xbf\xad]z\x86\x02Sw\x0c\xb1_q\xed4\x07\xf5\xb8\x8cF\x96\x8e\n\x81T\xe3\xaa$\xf3\xc9\x87\x92\x18a\xbaI\x1eW\xb2\xff\x19<\x89\xa3\xc70\x94\x92\xc8rk\xbf\x8f\xdf\xb4}\x99\xb5\xda\x07\x18\x95\xd4m\xac\xc2\xf9\xf5\xef\xael>\x9a\x89j\xc7\xe3\xadF\xe1\xec\n\xd6\xc4\xeb\xe8\x82\xb9\x05]\xa0\xb9\xcd\x1e\xa8Z\xe1W\xec\x80#\x05\xd6A\x01N\xbf\x907\x84F\xd5(\x07\xb8\xb6\xc8\xc9wc&\xd0\x924\x17\xea?{\xd8v\xc1\xb4\xcc\x1b\xed"\ryx\xfe\x91$\xde\xb07Z\xba\x04e\x93\x92^\x8cm\xb1I~2\x96\x944\xb6\x1a\xbd\xc7\x0c}\xc5_\x9ah\x8f\xa1\x00k\xa5\x1f\xf8o(\xd6\x90\xd5\x1bk\xec\xa1\xfd\x95P\xa4s\x83\xe9\x16\xbb3;y\xf0$A\xc2$[;\x1a\xb2\xa3\xb8W\xcdjX\xfd\xbf\xfa2|\xfcy\xc3\xa2:@\x962K\x9f=j3\x87c![\x90b\x07h\xe5\xa9\x19\x04l\xfd\xb3\x96j\xb4\xa5XNF]\xcb\x0b\x8bb\xebs\xb8\x98M\x14\xaa8\xac^\xfd\x8c\x1b\xdf\xb1>\x97\xf9\x8e\xcc*&\xa4\xfa\xfd\xcb\xe1\xe3u\xaem\x8c\xa4\xa62\xfa(j\x11\xab#\xce\xec6\xe3\xc7\xc4\xf7\x8ar\xa2\r\xc5,v\xa9\r\xd2U^\xaa=\x89\xa2\x14\x14"s\xb9\x18Z\x94=\xf5\x1ar\x8a\x01#\xca\\ui\xc4\x19Yj\x84\xb7\x1f\x8a\x05\xa0+7)<\x86\xb8\x8b\xca\x91\xf4\xd9\x1c\xdd\xa2\x96\xe0\x18w\xd5\xb0\x99C\xdd\xef\xfc@m\x85\x9fE1s6xq\x1b|h\xf4\xbe\x16\xab\x94\xaf\x04\xac\xc5\xa2\x03\xa4B"@\xe0\xfe\x0f\xf7w~\x127\xbf\xe1\x8ch\x8c\x04\x1a\x07D\xeb\x12\x8e\xfa\xfe{\xae\xa5\x07\x8f\xcc<\x17\xb1\xcf?;-%YF\xd5\xbd\xda\xb7s\xa2\xc4t\x81\xcd*\xa6o\x91p\x1db\x95\x16|\xa2\\k\x91\xf8U&\x91<\xc3\xa89\x88\xa2\x02X\xd9.\x1eu\x16\xadj:A\x17\x19\x13#\xb6l;6\x18\xdcz\xd3\xf7?\xab\xdeX\xc0\xf4\xc6\xebS\xaaNV\x84Y}\xfc\xea\xc6\xcb\xeb\xca\x19\x8dLl\xacE\xa4\xf8b\xdc\xc4\x16f\xc4\xe4\x00\xb8\x8ey\xea\xa6\x8f\x9d;2\x8b\xfeDa\x86\xcc\xef\xa6D`\t\xd7\x9eT_\x1d\xb34\'$\x15\x84M\xef\x95J&U)\x98\x1d\xc5\x19\x19\xc7\x03e\xd3jg\xa8\xa8\xd8\x93\x95\x06\x87\xa7\xa3\x1e%\xb0Zj\xcf1\xdc\xbb\xb1~\xa6\xfar\xded\xce\xc8\xe7\xa1\xc4\x882L\x18\xe3E1\xee\xdc\xb2S\x08\xd2\x9a#\xbbO\xd8\x05\xcc\xe2R\xc4-\xe7\x80\xa1\xc6\xa3\x06\xd8\xc0\xf6\xb1t>\xa0\xc3\xc3UZ#\xb3\x96`\xd7\xf9\'\xaeh\x95x\xc8\n5\xfe<\xe2\xbc\x13gZ\xd9\xd4\n\'\x9f\x85K\xa74C\x827\x03J\xde\x8cQ\x0f\xdb\xf9\n\x81\x8a/\x8bL\xae6z\xae\x816\x9c\xd8\xf2\x88\xd7\xea0%\x89A\x85\xb5\x96\xf1\xc98Y\xb6\x17\xb1\xe8?f\x0fR%^\xfd\xfa\xd5p\xf2T\xed\xa7[\x9c\xd1\xc4j\xe8\xb9\xe5\xcdSk\x1b\xaf\xeel\xde^\x8cF\xcf\x10\xcc\x8cz\xd3@\xc4\xc0\\\x94\xa5&0[\x08\x10\x8e\xf9b\xb6\x86\x1eR\xc3]\xe7\xeeX\xed\xa18\x9bn\xa1\xe0\xe4\\ \x94\xfc(\xbb\x99\x88YGE\x84\xe5\xf0\x9a\xb8\x9a\xda\xddD\xa8\x9e\xce\x03z\xcdT\x93I\x1c(\xc8\xc9\xe9\x94R\xde\xc9\xe7\xd8\xbay~y\xf3\xc1]s/\xa6\x13Z\xf6\x19\xee\xa3\xed\xc72\x9at(\xd0L+Z\xda\xad\xb3\x9a\xbe]}K\xad1\x16i\xb1\x02\xbaSLm\xba\x7f\x8f\xa9\xad\x89Y(\x94K\xc1P-\xf3\x83\xc8@\xd4\x02\xdf\x9c\x11$dH\x12^:\xd5\x96\xe7\xc2\xf1s\x11\x0bEU\x8b\xbf\x9eODs\xb9A\xb2\xb3@\xbdY\xda\n\xf0\xd8\xedl\xd6)\x955\x1bQ\'b\\Sq\x08E\xbc\xa9\xc9\x89\xcd\xc5\'z\xb8\x11K\xb4\xa8\x15\x9a\x1c=^\xdbTM\xcc!\x86_u\xacQ+)q\xdcS\xcc\xe6+9G\x07yZ"\xa5K\xb9\x87Q\xac\x1en\xd5U\x1eCn\x0e\xe4Sci(h\x1dP\xa0\xff\x1c\x9f\x10M:=U\xfb\xf1U\xed\xe9\x93\xfa\xb5\'\x9bW\xe6\x04n\xca\xe5\xd6~\x9a\xc5\x9d\x10\xf2\xb5\x00VH\xe1\x8cf\xa6\xadR\x04\x10\xd2\x7fH\xe3_\xbcA\xda\xc0\xa0\xefUJ\x04>\xe7w\xe9o\xb6)\x14\xfc\xa8\x9f\x0c]\x9bb\x80S\x99\x8c\xdeq\xf7\x89\x12\xee,f\xc7bfD\x071\xd9\x12R\xe6\x92\x9aQ\xe5\xfa\xe4\xd4\xe6\xbc2\x02\x1a;7\xdc?\x1b\x7f\xb1u$\xf0\xc4{\x7fr\xa9~\xf9\x1cV\xdfGQT\xb4;\x80\x18\xaf\xde\x8c\x1f(0z\xe3\x01\x92z\xdd\xba\xdeqw\xdc\x07\xd6\x8dp1\xd3h\xc8\x0e\xe2\xf1\x87V\t\xf0E;\xc7cm\x0c\xa81Ej\x86\xd2|\xc4\xca\xd6\x0c<EK5:\xcc\xca\xe45\xa6c\xe0\xd1\xa9\xb3o\xd9f\x7f\x86\xa3\x15\xfd\x9a\x0bH\xd0y\xbd\xfc8\xde\xd4\x1d\xc1\xaf\xa8[\x0f\n\xe1\x8a\x00\x12\xa1|\xbc,M\xe0\xd3J\xa0\xc6\xe8D\x9d1\t\x9d\xc0\xb1\riSX\xb8Y}tG\x1f\xa4\x89\x04mw \xd4\x17\xee\x85sW6^\xbd\xaa\x9e\x9e\xab.<\xa8\xdd\xbc\x17/\x8b6\xa9\xbf\xadRB\xa3 \xfd\x0b\xe1\xef(cK\x85\r\x94\xfdO\xa4\xfe#b\xdf\xefb\x9f`\x14@\xf0\x9d\x827\x8c\xad@>\xfc\x13\x89\x98\xa2\xba6\xb7thy\x85\x93\xa3\x02\xc3\xd4\xaa\xce\xacc\'\x04i?.:\xaa\xcdLGr\xf5\xdbW\xf5\xabw1\x19z\xf1\x05\x92r\xb2\xf4\xbbR<^\xf4F\x8a\xc2\xdcq\xf5\x9d\xb1Z\xf5\xfe5\xcc`\xfeSG\xa9(\xbd\x84. 1ka\xcf-5\x9cJ\x06W\x97\x97]z\x96\x14\xe6\xa8}\xce\xcc\x02\xdc&\xd5\x8a\x13c\x04\\\xb1^\x0e\x9c`\xd8g\x8c\x1d\x85l1\xb7&\x1b\x01]\xf7\xbb\x06]\x97>\xd9\xc8\xb4c\x19\xcb\xcc\x0c3Kpl\x96*\xadd\xbd\xcc\xd8\xefx\x10\x01\xceFk\x9c\xe2\x84\xc2\nQ@~\xb9X\xc3hX\x16\x03\x8c\xc2\xab\xbeP\xf1\xd9@\xc8`\xeb\xd3\xf8\x15\xd8\x9f\x0e+\xb3n\x8fMa\x99\x9e\xe2\xf6\xa3\xfa\xd4\x9c\xd9\x82C\x8d\xd5\xd1`\xae\xffm\x9d\xa4\xf2\xe5\xb1\xffs\xe5$\x97-\xd33\xd5\xad&\xddn\xf0\x84ku\xb97\x15\xfe\x8eM\x00\x90\xbf\x9bL\x00\x80O\xb4c\x0b\xb2z\x00|\x9e!\x15;\xfd\xbd\xb5\x07\x8c\x9a\xc1b\x94\xd9\x8df\xd6e\xf3.jW\x17{j\xdc\xf2(\xc7\xb1\x81,\x12\xb3E\xb0\xa1\xf1n\xd3\t~\x98\xb3\x8a\x96\x0b\x97\xbe\xa9N?\xab=\\\xde\x04\x1be\xfa^\xed\x1br\x82\xf8%\x9c\x93N\x86\xa7\x11\xbc#\xbe\x9d=n\xac"^\xab\x9bG\x88\x02\x07y\x8e\xec6rS5\tK\x834\x9d\xd6\xf89x\xdb\xe6\xeb\xb5\xcd\x17wMh(\nR\x1d\xffVzD\xd7\x9e\xd4o\xbe\xe6\xda\x19\xe9\x85\xd3\x93\xc1F\x82x\xff3\x85>\x85B\x00c\x15\x87\xfa\xd1<\x00\r\x8b\x97T\xb9"s\x96\x1e\xd5\x07\'\x18\x90\xa4\x11\x02\xa2\xe1\x8f\xd2\x0fA\xbe\x12[\xb6:rX]\x9d\xab^\x98\t\xa7\xbf\xae.\xdc\xe4\x99}\xd1\x98\xc0\xb2?\xba;Y6\xe4%\xabp\x94\xd8\xd0\x10\xf0\xfc\r\xb0+\x1d\xbb\xa0\xcb\xa5\xc6\xef\x80\xe8\x8c\xaaBX\xbdFX2\xdf\xbc\xb1\xf2\x03\xe0D\xf1+!E\x0f*l\xcbR\x82\xa0/2\xf4\x81\xcaG\x82\xdd;vtt\xbe\xdd\xd6\x0e\xff\xd7\xb1\xfb\xe4\xa1\x83\x87\x8f\x8c\xed\x18\t\xfe;\x9fyO\xae\xeb\xa4>\xfd1#\x96\xf9\xa6\x16\xab\x1fb\x83(\na\x17\xcc\xa2\n5\x03\xdb\xb9\x00[\xe7\xa4\xd5\x9d\xb8\x16\xdd\xdb\xbc\xc3\xb3\x13\xb0\x1e\x06\xebe\xa8zg\xb0\x92\xc7C\xd0\x9b\xa5\x0e`t1\x80Xi\xd0!po\xfd\xf2\x97\xe8\xba.]\x03<\xb3\xb5\xc4f\xb9\x1e\xfb\xc033j\xe7\xc6\xab\x93\xe7\xa2\xcd\xb7Qg\x19\xc1\x9b\xee\xcc\xa8\xa9\x94n\xb1bv%F}h\xf1\xb1\x8d\xc0CFQL\xd4\x8d\x06\xff\x8d\x10\x11\xce\x99\x1c\xb8i\x9f;\xd34\x95J\x14\x99\xba\x1b\x11"\xe7\x04\nv+}\xb0\xb7\xdb\xf7=\xbf\xc58\x16}\x92\xdd\xf4\x1dLHD\xf5\xc6\xfeq\xa1\tV\x88\xa0\xa4\x11\xae(,\xc4\xd5\xd1e\x1b\xad\x07\xf0\xc7\xf3v\ttw+\xee\xda\xca\xb9A)ok\xcc\x82d\x07fb2aK\x01ki/\xbe@y`L\x9d\x0c\xa7\xaf\xb0\x8c\xac=\xfb\xb2>q\x81\xb1\x1f#(\xe62m@\xe6\xf1\xdd#C\xa3\xa8\x89h\n\'H1\x1c>\xc6\xe9\x1c\xf0\xe7\xfb\x1d\x1a\x99\x10\x05\xa1\x1e<\xc0J\xe7\xcf\x80\xb3\xc6\xeb/We\x16\x1f(\x87\x85S8\xe0rn\x9d[\xf2kW\xd7\xc3\xe93\xd2\xf9\x8d\xbfd\xc1\x9c\xe2\x030\xa7rR\xfb8J\n0\x83\xc0\xfed\xf0\xeb\xd7.T\x97\x9e\xa2\xf1\x1d\xdfG\xa3p\xedu\xc4\x04\xd6\x92)1\x895>G\xd6\x98\xfc\x03/c\xfa\x03I\xb1\xf1\xf2.j\x95\xefnF#l\xf4t\x1f \xa8\x128d\x8e\x99/(y\x81\x910\x90\xde\x99\x16\x1a\x10\xdbu2\xea\xa5\x11\x8f2\xa5\x99\x10\x9eE\x0c\x19\xf1cGg\xbb\xf4\xe6R)\xaa[\xe4\x8et\xb5%]\x80M2\x80\xa7W\x80\x10\xcc;\\\x8em\x0e+\xa5\x8d\x19\xfd\xb8\x1c^a"\x07,b\xc4\xe8\xfa\x8d\xf0\xd6\x13\xc9\xb9\xf1L\x8a\x07w\xc3\xc7\x17@P\xa3\x1d\xaf\x86\x97J\xac\x13\xad\x05\x85\x806\xef\xb8\x16\x8ay\x1c\x01g\xe3\x84!27#y&\x90\xfa\xb6\x1b\xa8&bJ\x18\xfa\xad\x01\xa89\xcbA\xfe\xd1DM\xa0\xb04\x10\x11}e\xa1~y\x86\xed\x9c\xa8\xc0R\xafnx\x16#\x01\xdbOe\xa2\xe8x\xee\x1d\x10#\xa3\xadD\xc1FB3V\xf12u\xaezq=&\x8b\x93\xa1\x0f\x10CY\xdcj:\xd3P\xd1(\x84\x02N\xf20\x07\xc1P:\xe6\x90\x99\xf3A\xbc\x881\xa2)j\x0e\xd0f\xa5\xe4KX\x18\xfe8\xb5\xb9xV\xbd\x1a\xb1\xe7X\x87q|S\xc1!\x91\x93\xd6\x87\x80%/Gw\xbf\xd9\xde\xde~Lh\xa6\xe0Pq\xb0QS\xdd\xb3\x0f%x\x83f\x94\t/\xda\xcc\x90qx`\x9bQ\xb4\x81+.\x89%`\x85(\xc3\x06N!\xc7\xb4\xd8\xfc \xea\x8aH\xe5\xf9sq\x8f\xf8\xddlghh9c\xa6&o"\x07\xf0PR\xcc\xf6!\xf9+\x1a\x91X\x08)\xfb\x9c\xf5\xa9\xd7O\xb1A\xb6\x01i\xecB\xf8\xfc\x07N\n&\xa0`\x0b\x11\xdfO\x19Dx7\xe2\xac\xad\xe0\x01ixE7\xcb\xe9:\xc9\xae\xeb!\xbf\xbaM?6\xe8\x97\xca\xa5x2\xb11\x90X\xcf>\xe3\x14;I\xdb\xa8K_3\x0c\xea85\x1d\xadq\x021\xcblB\x13\xbe\xa9\x8b&\xfc\xa6\xcb^\xd9\xcew\x91\xcdI9\xedlW\xaaz\xe1\x8b\xfa\xb5\xb3?\xaf\xceq\x12\x975\x11\x88\x16\xb0\x1d\xcb]\xd4R\x96\xc2\x02i{\xd8\x89d\xa0\xa6I\xdb\x1ap1\xe0\x01\x87K%u\xbe\x93\xf5|.\r6\x06\xda\x18\xda\x9c4\xb9\x18\xdd2\t{a\xbc\xf6T,\xee\xdaO\xe3\xf5\xf3\xa7\xd1\xbb\x00;\xee\xf2\xaada\r\x9dMB\x82\xbb\xc8\x95\xf5\xc6\xf5\xe5jN2\xdag^\x9e\xc6\xbd(\x98H[\xe3\x91G|\xe7\x93\xd8\xc3D\x08\xcaWi/\xe7\xa6\xd7\xe8K\xac-\xea\xb7^\xd5^L\xa0> \xda\xc4\x8a5\xc0\xfe\xcd{\x9b\xdf}c\xc2\x8c\x80Qp_\xde\xdaG\x1b\x050\xc1P\xe2\xf1\x8aX\x7fl\x95\xfdJy\x08QBq\xdf\x80\xb4^4E[\xb3\x8d\xaak`\x05)\x14\xa3\n\xad\xd8\x7f\x02\x8e\x93\x11\xa2\xd3\x8f\xf9\xf4e\xec0)2\xd4\xd5?|\x81e\xbfL\x0cW?\xdbxE1\xd8\xf5E0\xd5#\xcb\xa74\x04\x82\x8b\x9b\xc5\xb1\xb1/\x95H\xa1b\x18\x08\xbb\xe6RL\x1b\x89>`\xfc\x14\xbbKSB\xa8\x9c\xe2\xa2\xf4P\xac!\x9c\x9b\x06SL\x9d\x1b+\x8f\xf0w\xd4\x8c\x97\x92\xb1\xbd4\xc5\x97\x9eb#]\xaa~\xf3G\x10M\xeail\xa1]\xf4\xab\xf9i\x9dQ\xa3_\xbdIOO\xff\x80\x91R~J]d\x7f\xb5\xdd\xb2L\xa4\x8b\xdcs>\xf8m\xe8V\x8f\xe4\xa4\xa9\x07\xe7\'j/\xcf\xb3e\x19\x99\xd4\xa0E>\xfbQO\x8c\xa4j7\xa4a\xe9\xc5\xd1\xf4`\xd8\xf4\xd4D\x0e\x07\x89\x1c\x81\xb4\xc9P\t\x8b\xb4p\xd5\x96\xb4~8l\xf0!$\xac\xacu(\x81`\xe1A\xa5\xb5\xeb?\xc1\xb9\xf3\xb4U\x9cMD\xad`,\xd8q\\\xc0\xf8\xba\x88(R)CXW\x85\xbck*\xad\x8f@e\xa1\xb5M\xd2g\xc4\xce\xe7[A\xef`Q\xbaKuvz\x009\x16\x8d\x14\xd8\x06\xe0\n\x8b\x88\x1c\x8dx9\x0f\xaa!24\r(q\xc4\x1eO\xf1\x9f\x00]\xed\xd23\x19\x19\x99\xb4\xd7(u\xc6\x06i\x13\x99j\xb5r\xc1\xc6\x1b\xc9A\x03V4\xe2\xdd6\x98WX\t\x157\x1d\xa8:;\xb0\x0bh^\x9b\x98\xeaA\xa2b\x883c\x12G\xa6Q\xaa\x12<\xd3+\xe3\x18\x19N\x8b\x1a\x96}\xe4\xc7\xa9\xa5\xd4\xfbM\xfb\xd7\x08y\x1b\x92^\xce\xa7\xa9\xf1\x80A\'\x0c\xdf[CpJ\xe5\xa1\xd1\xc6L\x8b\xb1e\x99\xb2E&\x05\xcf<`@\x1b\x81\xdb\xc6\x9a1g\x1e0\xf9Y\x8e\xf23$q\x0c\x1cS*%\x07\x92\xc6\x7f\x1f\x01\xc5 0A\xa2\xd1>5\xc3\x03\xbe\xa2\xab\x0e\xcc_j\x8e\xd9\xea\x80\x13\x9e\xafGe<X\xa4\xe4\xdb\xc5\xc0m\x04E8\x87\xa6\xf4\xb3\xfc\x11\xf59}\xd9\x14\x97\\\xca\x93\x84&~\x8e#\x18Q\xcc\x0e\xa73o$6\xbb\x07\x8c\x08\xd7)\xcbH|\xd5\xb2H\x92\x95q\xec\x17$%\x80\xe86F\xba%\x80\xe4\x11\x96\xac\x84\x17n\x02\xbb\xb0\x04\xe7\xe3\x0b\x9f|\xcf\xe7[=5Y\xfdq\x82\x19\x8b\xd3ilk\'\x81\x17\x0f\xb2\x89\xdfx\x84\xcfU\xbb\x8f\xcd0\xae=_\xa5\xaf\xd1B\xd2\xd2R\r\xa7\xa3\x92\xac\x98\x94J\xa4\xd1\x8c\xc3\xc7i\xd0/VA\x80\xc9\xa4\\\x16\x9dZV&\x7f\\\xb2\x83 \x89h \x03\x9c~\x84NN\xd6\x01%\x9a7t;\xd3\x03\x16x\x15\x1d\xf4!\xc1\xe4\x00\xbd\x06>\xad0\x93Y\xc5\xa8\xe2$\xec\x93_\xabO|\xc94\x89P^\xbfQ\x1f\x1fG\x95J\x90m\xae>\x00=o\x86\x93\x9a\xd3\xed\xbf\xe2\xae\'\xf7\x16\xf9\x10\xe8\xe2\x90\xb7\xbe\x8d3a\xec\x83<\x18"\x82\xe9xt\xae\xd1\xbb\xd8\xca\xcb\x88\xaf\xb8\xcf\xe5\xd7\x0fT|\x00\xc7\x97\x08\x8d.\x1a1\x91\xaak\xc0\x1bW\xe1\x00\x82\x9e\xe8\xa7\xed\xf7\xed\xa0j".)!\xfd\xef\xbd\x07\x0fp\xe9\xbd\x04N\xb1\xef\x01\x03\tZ\x1e\x80;d\xff\x1b_\xdc\xe0\x8c\xa8\xf2k\x0c\x1d\xf5\x83\x9e\xf7\x9b\tNZ\x93\xed\x1da1L\xa8#\x84\\\xc8\x8a\x85\x1f\xb7\xe67\xef\xafI\x19\xcfg/7^\x9e\x87/l\xac\xcd\xf2\x17\xb6\xceX\x8b\xd0h\xc1\xc8\\&)[\x0f\xd9>z\xb2M.\x14!\x1anHE\xd3t\xbb&\xdfn\x14\xebD\x89\xc4\x07]\x14\x18hCY\x14(h2\x8d\xec$UC\xfc\x9b\x92=Jqr=\xb2/\x08*\xe4-\xe9\xac\xe6qg4Hv\xc9\x98\x8c\xce\x10)\xeca\x17\xbd\xf8s\x13\xe0j>\xe2\xf4&gxp\x1c\xd9\xd2\xb5\xc6br\xaa\x18\x86\x93\x84\r02(\xf7\x8aO\xb8:\xed\xe4X\xe3&\xa8&F\xd3\x81\x8c\xd5\xf4\x88p\x85\x7f0\xdd\xb9\xa5\xaa\n_\xaf\xd5.\xdd\xab\x7f\x1d\x1b\xb5\xae\x895\x8a\xdd\xb2\x1d\xfd\xfc\xf9\xc6\xcb\xcf\x92\x10\xc3\xe1#\x88\x0cl\xe4wRu\x82\xfe\xb3\x19\xd3s \x1c\xf4\x7fv\x083Me\xb4\xb7b\x91%r\x99\xf1\x02\x92\x86\x1a\x05\xc1;\x85\xc0\x1b\x93\x8c\xd1\n\xf1~\xd0\x04\xd4\x06\x8e)\xedB\x10\xa7\xf4\x1bS\xa4\xb4\xa2\xadQ\x8a\xbe\x89\xf4\x92\xb2\x0b5\x11M\xfcY\xd4\x9a\x80\xf2!\x1e\xf6\x8a\x89o|\x1b\xc7\x92\xa2\x98_\xdb\x16\xab\x99\xf3\xd1\xa4LD968\xb9\xee\xeau\x9c"\xcfa\xdcf\x86dL\x95`\x94*p\xca}\xe8\xef\xfa.Qs\x9a\x1d\x1f\xda\x17\x96e\xe0&\x8f\xf2\x06\xa9\x00R#\x13\x1b\x01}\x87\x06\x02v\xed\xb7\xf3\x81\xd3Tbb\xdb\x90\xce\x88\x93\xd3\xa6\xe2\xafZA\xa9\xfaN\xe5\x81\xc0\xb3Bs[\x05\xad\x14,z\xa4X\x95\xf6\xdfhj_4\x08\xdd\xb4>\x7f\xf5\xe9R\xba_\xb1J\x8ag\xf0\xd0\xbf\xfb\xb0\x7fr\x9b\xb3\xdd\x8b\xa997\xab\xccV\xba9\x85\xf7IW\x85\x04\xd2\xfa\xc0Ev\x14\x16ba\xb2\xd5\xe9\x82\xfb\x84\x15\x0e\x86\xed\x0c\x0e\r&|\x17\x16Y|\xa0\xce\xa0\xd8Ut\xbbH\xd3\xa3Ul\xdfEd\xfa\xc6\x16\xaf\xeb>Q\xc6\xe20mv\xf3\x19 \x15\xb6Rz_/\x83\xb2\x0fC\xe3\\-fq]\xf6V\x9b\xe0\\\xae\xc4\x94\xb9@\x96\x7f\x06\x1a%\x9c\\a\xf8\xc3y\x95\xf1\xdfv#[2c\xe2\x84\x9a\x1f\x90!\xcf%]\x0f\x82C\x89\x80T\x10M\xc0\x8d\xb7\xe059\x17}\xa7\xc7\xa3\xbb\x86\x0c\x01\x07\x97\xa8R\xd5kmEt[\x1cK$=\x02\xb3\xae\x93\xe6\xde\xe8z\x9aF`z\x06\x8bHf\x91\xc9_p\xa9\x80\xa4\x7fT\xcf\x05\x01\xc9\xaeo\x17\x92\x02\xbf-6\xa6\xa5\xbc\xb1\x11S\xca\xe3\xac\xb3\xc9hj\xf2/\n\x1580\x1a\x05\xa2\x9c\xe1H@\xb6X\x8d\x0fI\x1dd\xf8D1\x18\xf2K\xa7h\x04(%v&#\x0c(~\x97\x98h\xbc\xfd\x81\xd2T\x083rq\x0b\x8b\xa2\xf0r\x01\x92\x9e\x98\xf1XYK\xde\xe4\xb3\xd5\x82R\x8aW\xb0O\xb8\x85\n\x06\xe1\xe2[\xa5\x8fQb\xc6\x1f\xd3H\x8f7\xb6\x81R\x17uR\xdf<\xc8\x19\x92*Xt!\xc1\xfd\xa8\xe9U\x0c\xd0(S\xb5\xc5\xbeU#\xbc\xb8{du\xd0\xc5t\xd8\x08\x8f\x13\x9c\xc7W1~zf\x96\xe3\xa4\xb5K\x0f\xeb_\x7f\xb3\xdd\xd6\x9b\xd6$\xa6\xddb\x19\x13D\x9e]\x96\x999\xa6\x15\xc8Xj\xf6-\x85\xc0?[\xed[\xd3\x82\x94tx\xf9\xbc(V5\tK\x95\xde\x085H\x1b\xb3;0\x80\x18\xd3f\xba\x9e\x07\xbb\xfd\xfaR\x0b\x1c\xd7:r\x03\x11\xd1\x03\xa2\xe9\xf4\x03@\xa8\xa8\x1fV\xc9@3\xdb\x93\x8a\xd6\xbc\x0cd\x17\xcd\x84\x95}oC\x0c\xa2OK\xba\t\x04\x83\xd7\xea2\r\x1d(\'\xfah\xb1\x8a8~\x02\xa45\xba\xa0\xd2}+\xe1*\xe7\x97\xb6\xcd\xd72I\xd8\xdc\x08\x98G\x02aeV\xcd\x82\xc0x\x7f\xf5\xc2\xe2\xc6\xea\xe3_\xb5\xe1\xa2L_n\x07Fq\x8bx\xf8R\xad\xdab $\xf3\x8b8\xe8\xa1\x1a\x8d\x01.\xbe\xe2h\x82\x9d\xcdV\xb86\xe2\x97\xb6\'\x17\xe9-^EY\x06d\xcf\xa1\x0f\xb2\xa0~\xd5\x1e\x92\xe6R,x\xbb\x95=d\xce\xb5\xc0\xa1Nt\xd7N\xd9\x88\xb1\x0e\xda~.\x8f\xa2\x1c\xec\xdc\xe6\xe5\x1f\xb1\xee=\xb0\xf3\x1e^6\x03\\<l\x9d\xc2 h\x98\x9fZ3s\xfcZ\xc8m\x15Z@\x1d+\xf1\xc5V\xabI\xac\xfe\xcf]\xd6\xae\xddMB\x0e~.yQ\x96\xde\xd2\xdf+8\xfeD\xd7\x89\xe4*\x18\xc3E\xca\xd5y\xdbd\x90\x87\x88lc\xe5,\xde\x95EI\x04\x1d(\x95\xfc\x02\xa5\x1bx\xa2.\x17\x94`\xa5\xde\x93\x06\x9f\xa3i\xaeA\xb6\x97\xf4\x91\xc82\x17\xc8\xf3\xde`\xc0AQG\x0f\xf7\xcfa\xd37\x0f\x83\x90{m\xb0C\'h\x12\xa3\x12aA\x99\xea\xf0\xfa-}cMx\xe7\xcb\xf0\xecu=\xba_UM\x8do\xbe\xfe\x1c\x95,\xc5\xa5\xad\x0f\xbd\xc1&\x9e\x13\xe7{\xe9"\x80f.\x92\xce4\t\xfc\x9c\xda\xe5m`\xb4\xde\xa6\xb8\x08\x85+r\xae\r\x16\x03\xf0\x7f6hfX\xebl\x13\x00\xc5;`\x12A#m\xfd|\r|\xfb\xabw\x01\xd2f\xec\xd1$;\x9a\x8a\xa7\xed\xd5\xd6\xaf\xfdG1e\xfd\xd1\xfa\xf0\xe0\xfbm\xb8\xa9>\x8cY\xa4)\xa2\x1ft\xa5\xc0\xec\xcc\xdb\xd8\x15w\xb4u\xd7\x9b\xed\xed\xbb\x8fm\xe5\x92S\x8f\x14e\xf4\x83\xa6y\x8a\xe6\xeew\xecf\x08\x95\xadl\x9e\x99\x88\x9d\x01\xa7\xd3\x9aw&\xca{w\x9c\xd4~\xacY\xd5\xf3\x96\xeeP\xa4_\xe9\xef4\xec\x8bJ\xf2U^\xce\xc3:\x1a\xa4?\n)\xdaQ2\xa6\xf9\xe6\x84\x9d/R\xba\x8ds\x85\xb4\x0f84\xae>\xe4 ]\x13\xd2\xc2\x9d5\xa3)p\x07|\xccCs\x0eN\r\xbe\x15\xb7\xd7+r\x06&*\x81\xc8\xdb\x9c5\xd6\x05hM\x94\xe8\x02\xb8ew\xc2\xcf\xcf\xca\xfdw3\xb3\xd2\xd8\xa6\xad\x7fuO\x84D\x9d\x9b\xb6\tP\xfc\x86\xdf\xdde\x1d-pJ\x10=*\xdc\x08\x9bR\xe8\x05TbV\xb4\xca\x19\xf2}y\xb8\xed\xc2\xd1\xf6c\xa6#\xef\x15\xfbh\xd5\xd4\xb1\xe6\xc9\x13\xbe\xff\x95r\xf9\x80\x8a@\x9a#T5\x169\xb08\x99\xd0l\xdc\xdb\xd2\x8c\xb89\x8fc\xe6\xb9\x00_\xd59\xb1IA\x19\xfd\xf3x\xe7U\x93\xf2\x83\xa4+DH\xd82\xd0P\xf1\x07l!!\xfafd\xf3p\xcb\x17@\x8eu\xe6[\xfa:*\xba\xa0\x9b\xb7\xb0\x9e\x90\xa0\x8d\xfc\x82-\x0fh\x0by\x80\xce\x00\x83}\xb4\xb5\x03\xef\x8d\xc9D%\x13\xcd\xc2\xebl\xd4\xb3\xbb\xc6\xe1t%P\x14\xab\xcbU\x8c\xc1\xb6t\xc7\xa1L]NL\x8c\xa1z*\xd1\xbc\xaf\x02%>\x9d\xaf~}\x93S\x98\xcd6\xd4\x0f\x82\xeaxl\x06\n\x1b_t\x03A\x99$\xabv\xc1\xb0<\x9dj\x16\x94"\xa5l!\xa76P&\xa3\x91K\xdeX,\xde\n\xcc1O=r\xdf|\xad\x8b\xc6\xc4\xec\x8e\xe7\n%\xd4Fy\xce\xeaOW\xc0\xbb6\x01\xd6\x86\t\xd6B\r\xb8\'\xfa\x10\xe7 \xa9\x14,;\xac\xb7\xdaw\xb7u\x0c\x8cY\xe0\xba\xd5\xe7\xbf\xda\xd6R9\xcc\x9f\xc4\xef\xbc\xe1B\xfeV\x92\xd2\xc9\x02\xf6\xe9\xe78m\xf1\xc2M\xbd\x03\xa9R\x87\x87\xdf\x00\xad\x7f[\x9f\x9a\xe3;\x96\x81\xbd\x1b\xa0\x168\xd21\x10\xb8p\xa1\x8c\xb3\x8d\xb5l\x14\xcdl\x14\xecF\xb9@7V\xcag\xc2g\x04\x85Um\x1b\x161/\xce\xa2\x0e\xa0\x83\x8f\xec\x8b\x89\x0b\xe1\xf2\x99F\x91\xa9S\x82\xcd\xd2\x81\x7f\xa3\x06\xbe\xbd\x87>\xe6\xf3\xe6\x84\xa51\xc7\xc3\xaf\x14\x83\xe8\x1e]T\xd0\x8d\x1a\xea\xcb\x8d\xd5\x97\xa8+\xaf.\xaa\xcb\x810\xc5\x85\x18\xa4\xe3\xdeX\xbb\x88\x01\x14]\xe5\x02/kH\x0ebJ8\xc8;N)\xdd\x91y\xc3,y4\xe3k\xe2\x1c\xa2J\xe1Vd\x83\xc5\xa8W\xb4\x92\xcf\x93\xe6!\xb6r\x8c9\x83\xccL \xa1P\x1c\x18\x15m"\xd3\xd7\xcfc\x1b\xdb\xe3)]\x7f\xa3\xad\xeb\xa84%\xe6\x03D\xc6\xbe\xae}\xc5yA\x0c-\xc9q\xac\xce\xd4\xfe\xbd\xd1\x12#1\x08\xaa\xd2\xe0\x9b\'\x9a\xbc\xa2\xc1D\xe7\xeflA\xf5q\x8ag}\x1b+\xff\x14\x97[\x17\x1b\xb0\xa1j\x94\xde E\xe1\xc8\xbe\xb5\xa8\x00j;\xaeLo\x95\xc1\xcf\xfcZF5B\xca\x0c\xaf\x16C\x14\x0b\xd3%\xc0J\x96\xa3\x9d\xd3\xa4\xb07\xc9\x80\xc0x\xf9\x11{\x94\x8a\x7fX\x89\xc4+yG\\\xbe\xd3\x91\xcb\x905\xf7)\n\x02\x16\x91b\xb1\xda\xe9\xdb@\xc7rk(\xd1\x08f\x95\x95\xd6\x03\xef\x84\xcb\xfd\xb4\xc0\xc0{\x83\x9a%\xe7hsy]*B\xb6\x12\x85\x8f\x1b\xaexI\xc0h\x94\xea\x82\x19$\xed<\xe6m.\x92p#\x04\xc5\xd0\x91(1\xe4\xe2\xca\xb8\xe0!\xbd\xefJ\x03m\xd4gw\xee\x9e)m^E\xc3x$\r\xb9tj\xe3\xd5=\xde8Fr_\xce\xff\xe6\nJ.\x9f\x95\xca\xf6J\xbf&\xf0\xc6BJ\x9e1eF\x91\xb7\xaf\xa7\x04\x06\xf4\xb2\x89\xeb\xc2\x1cu%9\x88\xaf\xc4\x95(j\x9c\x8f+\x15\xfe\xeb7\x12\xf7\x9b\xe8\x817\xdc\xe4\x13\xb3\xbf\xf5\x95k4\x12\x11\xed\xb9\x127\xfeR\x10^\xcf\xf4\xe1W\xa52m>\xde-\x99N\xfd\x81\xdb\xc2K\x89)\x87%\x1a\xd8\xd2F\xfc\x13\xa0\x02N\xab\x19\xda\x19\xaan\xa4z\r\x12t(\xfbHg\xf3\xaex\xbe\xb6\\\xed\xfe@\x8d\x86[\xae~\xf1\x08\xf0\xc7\x17Ia\xcd\x16\x8db\xdc\nZ~FP5\x1d\xa4\x88\x93\xbbenb\x81\xfeQ\xf0\x86S\x19\x19\xdb2\x80\xa3Y\xc0A&+Z\x9922\\\xc8\x13 q\xf0k`\x0fs\xd9<\x17=4\x0eM\xc0h\xc0\x83\xbb\xe1\xf4\xdd\x984\x94\x16#\x02\x91.\x862o\x06\xe2\x9b1\xd7\xceT\x17\x1e\xe0\tM?\xe6\xf5\xb0\x8dH[\xd4\xbf\xe3\x13\x08~G&u\xfc\x9a\xda\xf8\xd5\xd8\x14\x83DO\x99\x86\x0f#o\n\x8a\xe5\x12G\x99\xd6L\x97u1)\xc8-8\xd4\x13\xc5\xae\x18Z\x0b\x9a2\xb8\x15\x99\xaf\x8a\x90\x90\x91\xec\xe3\xb83\xda\x95\xb7\x0b\xfd9\x9b\x06G\xef\xe6\xf1\xd1\xd1\x8c\xcb\x02\xcaV.F\xb7\x87\xa3\x890\xad\xd8zF\xc3\x1f\x98\x80\xe1x\xfeb\x0fb\xf5)\x86\xf29^\xec\xf9\x94t#5\x83B\xfd\xc6\x02\xd7\xcd\xe1\xa5G\x89/G\x83\xa9\xe4\xae\x9e}=\x87q\xee\xea\xc1\x83G\x90t\xa3\xab#\x87v\xf6\t\x16\x93\x03A\x12\x9c4\xaa\xa5\x89\x1af\x9e`\xa4(#\x1c\xbd\xf2W\xcd\x8bJ\xcc\x89\x92\xf8(\xa3Vj\x05h\xf8z\x99\xef\xa9\x8b\xaaOMBK\x8c\xd9\rg\x96\xa3+\xe4\xe5B \x0c2\x02\x0e\x0c\x84\xd08\xae\xe67\x1a\xe9\x9b\x83N\x9aG-L4\x96\xb8tLW\xb6\xcb\x10Y]\x80g^\xf8U\xf2]\xe0JA6\xdf\x02\xcdn\xf1\x03\r5\x83\x1c]\x18z\xff\xcc\xe6\x97\xe7\xe4\x92FMzr.]\x96\xb1-\x04\xe7\x00O}\x8e\xe2X\xd1\x95\xaf9\xec\x82\xc4\xabG0\xc0\x82\xfaJ\xfa\xc5\xc3\x97s\xc0\x91\x1c\xb0\xe2wG#J\x9b\xb4[\xe2\x8dt\xe0\x16eb\xda\x8e"\x02xE\xa0\xeeSQ\xefE\x851\x00\x96I\xce\xec\xa7\xc4\x89m\xcb\x89\r\xebh@\xac\x17\xb8,\x1blc\xba\x8b\x8f:\x88Q\x8c\xea;\x97\x8e9u+\x16\xf2]\xc3@1\xc9\xb7\x93\x9d-\xcd\x89 \xfd5\xa94q\xac\xd0\xe6\xc8G\xba\xa7\x88\x95,\xba\x9f6>i\x15\x04\xc9\xe4lmmic\xe5\x9c\\\xda\xc6\xe3\xc6c\xcb7;\xbf6\xd4\x0b\xa4\x13\xd2\xd1\x05V\xf4\x81\xd0"\x9d\xc4\xee\xf6\x9d\xb9\xb1\xf8\xa7B\x8e\xd2\xf1$c\xee\xa2\xb4\x133\x91\x84\xe0e\xf0T\x92\xafE\xa0H9\xaf\x1at\xa7g)k$E"\xa5\xc9\x905f\x8f\x16Y\xdclO\xe1\xe0?^\x9b\xda\x12\xf3<\xf9.\xcf\xa89+kg1c\xe0\x0c\xd9\xc3\xaeg\\\xb3\xa6F\nO\xb0\xe9\n\x86\x98\xdc!\xfb\xfa!\x8a\xec\xa8\x1aZz9Ys\xa0\x9e`X\xc6~^[\xe0\x9fZ\xbf\xc9\xae\x8d\x8bhl\x14\x0e\xca\xb1\x8b^\xe5*X\x19(\xa4\xac\\\xaaa\xda\xa1\xee\t\xb6\xb2N>\xaf$6\xe9\x1566\xa5\xfa\\\xebpn\xf5xz\xdf\x1cI\xc3\x05L\xd8\xf7F\xf7C7^\x01\xcb\xfb\x8b_\x02+\x93\xaa\xc4,\n8:\xab\xcb\x82\xb0\xf5#\x97\xd3\xc3\xa9\x93W\xaf*`%\x08\xc1r\x15\x07\xc4\xe1\xf1Sa?\xddQ\xb7\x17\xbef\xc1n6_>\xd4]A\x0clt\x0e\xa9\xff\xbb\xf0\xa5\x857\x1b\xbcKW\x90\xab3\xd1k\xf2R\xdc\xcb\x1e]\x83\xd0\xd1N5/\r\x1bOe\xfe\x1f\x19\xc3\xc0\xed').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['t2v', 'i2v', 'flf2v', 'r2v']
REMOTE_MAX_IMAGES = 9

'''
Keep job directories isolated between different notebook services.
各 Notebook 使用獨立工作資料夾，避免不同服務互相干擾。
'''
REMOTE_ROOT = Path('/kaggle/working/remote_api_jobs') / REMOTE_SERVICE_ID
REMOTE_ROOT.mkdir(parents=True, exist_ok=True)

'''
Share existing state when the API cell is executed again.
重跑 API Cell 時保留原有工作及鎖定狀態。
'''
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('REMOTE_ENGINE_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=32))
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(2))

'''
Results remain available for up to 24 hours, subject to an 8 GiB storage limit.
結果最多保留 24 小時；若超過 8 GiB，會提前清除最舊結果。
'''
REMOTE_RESULT_TTL = 24 * 3600
REMOTE_RESULT_BYTES = 8 * 1024**3
REMOTE_MIN_FREE_BYTES = 2 * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = 2 * 3600

'''
Report only checks that can be verified without loading another model.
只檢查不會額外載入模型的服務就緒條件。
'''
def remote_backend_ready():
    if REMOTE_KIND == 'image_multi':
        return bool(globals().get('QWEN21_STATE', {}).get('pipeline'))
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        return globals().get('pipe') is not None
    if REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        gpu = globals().get('GPU_ID', 0)
        process = globals().get('SERVER_PROCS', {}).get(gpu)
        return bool(process and process.poll() is None)
    base = globals().get('BASE')
    if not base and REMOTE_IS_LTX:
        base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
    if not isinstance(base, str):
        return False
    try:
        with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=3) as response:
            return response.status == 200
    except (OSError, ValueError):
        return False

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'backend_ready': remote_backend_ready(), 'stalled': remote_stalled(),
                                 'running': running, 'waiting': REMOTE_QUEUE.qsize(),
                                 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return

        '''
        Reuse the same job ID for retry attempts after an uncertain response.
        網路回應不明時，重試會沿用同一個工作 ID，避免重複生成。
        '''
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        if not remote_backend_ready():
            self.send_json(503, {'error': 'Backend is not ready; execute model initialization cells first'}, 20)
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files = {}, []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name == 'images':
                        files.append((part.get_filename() or 'upload.png', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and files:
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (1, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names = []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value

            '''
            Resolve -1 once on the server so the exact value can be reproduced.
            在服務端一次決定 -1 的實際種子，方便重現結果。
            '''
            for seed_key in ('seed', 'lm_seed'):

                '''
                Ignore missing seeds and validate the supported 32-bit range.
                未提供的種子沿用 Notebook 設定，並檢查 32-bit 範圍。
                '''
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')

                    '''
                    Translate -1 into an explicit random value before inference.
                    推論前將 -1 轉成明確的隨機數值。
                    '''
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)

            '''
            Save supplied seeds for reproducibility in the job status API.
            在工作狀態 API 中記錄請求所使用的實際種子。
            '''
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        '''
                        Point R2V to this job's uploaded reference images only.
                        讓 R2V 僅讀取這筆工作的參考圖片資料夾。
                        '''
                        namespace['R2V_REF_DIR'] = Path(image_paths[0]).parent
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:

                    '''
                    LTX-2.5 uses a shared ACTIVE_JOB configuration.
                    LTX-2.5 使用共用 ACTIVE_JOB 參數。
                    '''
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'

            '''
            Clear old output metadata so a failed generation cannot reuse another job's file.
            清除前次輸出狀態，避免本次失敗時誤取其他工作的結果。
            '''
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)

            '''
            Execute a fixed local generation cell, never user-supplied code.
            僅執行 Notebook 內固定的生成程式，不會執行請求傳來的原始碼。
            '''
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):

                '''
                Include YuE2 replay metadata and score with the single MP3.
                將 YuE2 的重播參數與 ABC 樂譜一起提供下載。
                '''
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None

            '''
            Remove only source files bearing this remote job's unique prefix.
            僅清理本次遠端工作專屬原始輸出，保留 Notebook 手動生成的檔案。
            '''
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

'''
Keep one worker and one maintenance thread even after rerunning this cell.
重跑 Cell 仍僅保留一個 worker 和一個清理 thread。
'''
if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()
if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

'''
Close only the previous ngrok tunnel; preserve the running API and its jobs.
只關閉舊的 ngrok 通道，保留正在運作的 API 與所有生成工作。
'''
old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)
NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(Path('/kaggle/working/ngrok'))
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

'''
Always request the auth token again when this cell is rerun.
每次重跑均重新要求輸入 ngrok Authtoken，不顯示內容。
'''
ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')
result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')
del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

'''
Reuse the same API server when this cell is executed again.
重跑本格時沿用同一個 API Server，不中斷生成工作或清空佇列。
'''
server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()
if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:

    '''
    Update the handler on rerun without restarting an active GPU job.
    重跑後更新 API handler，但不重新啟動正在執行的生成工作。
    '''
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')
with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)
NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)
if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])
print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')


## 13. Web API 使用說明

In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request

'''
Request the API schema directly from the local API server.
直接從本機 API Server 查詢支援的參數與模式。
'''
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)
print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本地 requests 範例：06_HyperFlow_H3_Flash_影片生成_Kaggle_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')